# 실습5. LLM 파인튜닝 — 하나의 모델로 여섯 가지 태스크

**AI아카데미 [A4021] 언어지능: 언어모델 기반 자연어처리 실습 기초 · 2일차**

1일차에는 두 가지 구조를 만났습니다. 실습1·2·4A의 **BERT**(인코더)는 문장을 읽고 **라벨이나 위치를 골랐고**,
실습4B의 **pko-T5**(인코더-디코더)는 읽고 **답을 글자로 써냈습니다**.
오늘은 세 번째 구조인 **LLM**(디코더)입니다. 앞말에 이어 **계속 써내는** 모델이고, 지시문만 바꾸면 다른 일을 합니다.

그래서 이번 실습은 **하나의 소형 LLM(1~2B)에 LoRA 어댑터 하나**를 붙여
주제분류·개체명인식·기계독해·문장유사도·SQL생성·수학추론 **여섯 태스크를 한 번에** 가르칩니다.
앞의 네 개는 BERT·T5와 나란히 놓고 비교할 수 있고, 뒤의 두 개는 **생성 모델만 할 수 있는** 태스크입니다
(수학추론은 시간이 남을 때 다루는 보너스입니다).

마지막에는 **내가 학습한 모델에 직접 문장을 넣어 보는 데모 페이지**를 띄웁니다.

## 이 노트북에서 배우는 것

| 순서 | 내용 | 손으로 하는 일 |
|---|---|---|
| 0 | 세 방식 자리매김 · 라인업 · **모델 카드 읽는 법** | 읽기 · 모델 카드 열어 보기 |
| 1 | BERT식 vs LLM식 — 무엇이 다른가 | 읽기 |
| 2 | instruction 데이터 만들기 (6태스크, 평가셋 분리) | 데이터 생성·예시 살피기 |
| 3 | 토크나이저 · chat template · 멀티턴 · 흔한 사고 세 가지 | 학습 텍스트가 실제로 어떻게 생기는지 확인 |
| 4 | 학습 전 모델은 얼마나 하나 — 기준선 | 6태스크 평가 |
| 5 | LoRA와 QLoRA — 무엇을, 왜 얼마나 학습하나 | `LoraConfig` → 학습 파라미터 비율 확인 |
| 6 | TRL `SFTTrainer`로 1 epoch 학습 | 진행률 표 — 손실·토큰 정확도·GPU 메모리와 10%마다 태스크별 검증 점수 |
| 7 | 학습 후 평가와 전후 비교 | 표·그래프 |
| 8 | BERT·T5와 비교 — 같은 데이터, 같은 평가셋 | `bert_baseline.py` 두 태스크 실행 |
| 9 | 내 모델 데모 페이지 (Flask) | 브라우저에서 직접 입력 |
| 10 | 사례 연구 — "멈추지 못한 모델" | 출력 읽기 |
| 11 | 더 해보기 · 참고 자료 | |

> **진행 방식.** 셀을 **위에서부터 하나씩** 실행합니다(`Shift+Enter`). 각 절 끝의 **관찰 포인트**는 옆 사람과 결과를 비교해 보는 질문입니다.
> 코드는 전부 `task5-llm-ft/` 아래 스크립트(`common.py`, `train.py`, `evaluate.py`, `serve.py`)와 **같은 함수**를 부릅니다 — 노트북에서 본 것을 터미널에서도 그대로 재현할 수 있습니다.

> **GPU 메모리.** 기본 모델(A.X-4.0-Light 7B)은 **학습에 약 17GB, 평가에 그보다 조금 더** 씁니다 —
> 24GB GPU에 들어가지만 여유가 많지는 않습니다. 평가가 더 쓰는 것은 한 번에 여러 예제를 생성하기 때문입니다(§4의 `EVAL_BATCH`).
> **앞의 노트북(실습4A·4B)을 열어 두었다면 그 커널을 먼저 종료하세요.** 모델별 실제 최대 사용량은 §0의 표에서 확인하세요.
> `CUDA out of memory`가 나면 §6에서 `per_device_train_batch_size`를 2로 줄이고 `gradient_accumulation_steps`를 8로 올리세요(유효 배치는 같습니다). 그래도 모자라면 §5의 QLoRA(`--load-4bit`)를 씁니다.

## 0. 준비

이 노트북은 저장소 **루트**(`DeepKNLP-26.09/`)를 작업 폴더로 씁니다. 데이터는 `data/`, 출력은 `output/nb/`에 둡니다.
`task5-llm-ft/` 폴더 안에서 열어도 첫 셀이 자동으로 루트로 올라갑니다.

In [ ]:
import os, sys, json, time, gc
from pathlib import Path

# 저장소 루트로 이동 — 노트북이 task5-llm-ft/ 안이든 그 아래 폴더든 상관없이 찾는다
_here = Path.cwd().resolve()
_root = next((p for p in [_here, *_here.parents] if (p / "task5-llm-ft" / "common.py").exists()), None)
assert _root is not None, "DeepKNLP 저장소 안에서 이 노트북을 여세요"
os.chdir(_root)
sys.path.insert(0, str(_root / "task5-llm-ft"))
sys.path.insert(0, str(_root / "tools"))    # 미션·퀴즈 객관식(QuizKit) — 순서를 건너뛰어도 안전하게
import quizkit
PY = sys.executable            # 셸 명령(!)에서 지금 이 커널과 같은 파이썬을 쓰기 위해
RESULTS = "task5-llm-ft/results"   # 강사가 미리 돌려 둔 결과 폴더

import torch
print("작업 폴더 :", os.path.basename(os.getcwd()))
print("GPU       :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (CPU로는 너무 느립니다)")
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"GPU 메모리: {total/1024**3:.1f} GB (여유 {free/1024**3:.1f} GB)")

In [ ]:
import transformers, trl, peft, datasets
print(f"torch {torch.__version__} | transformers {transformers.__version__} | trl {trl.__version__} | peft {peft.__version__} | datasets {datasets.__version__}")

### 0-1. 세 방식 자리매김 — BERT · T5 · GPT 계열

이틀 동안 같은 종류의 문제를 **세 가지 구조**로 풀어 봅니다. 오늘 것이 세 번째입니다.

| | **BERT** (인코더) | **pko-T5** (인코더-디코더) | **LLM** (디코더) |
|---|---|---|---|
| 만난 곳 | 1일차 실습1·2·3 + 2일차 실습4A | 1일차 T5 파트 + 2일차 실습4B | 2일차 실습5 (오늘) |
| 하는 일 | 읽고 **라벨이나 위치를 고른다** | 읽고 **답을 글자로 써낸다** | 앞말에 이어 **계속 써낸다** |
| 답의 형태 | 정해진 후보 중 하나 — 라벨 번호, 또는 지문 안의 시작·끝 위치 | 임의의 문자열 | 임의의 문자열 |
| 없는 말을 지어낼 수 있나 | **못 한다** (구조상 불가능) | 할 수 있다 | 할 수 있다 |
| 태스크를 바꾸려면 | head(작은 분류기)와 손실 함수를 바꾼다 | 입력 앞에 붙이는 접두어를 바꾼다 | **지시문(prompt)** 을 바꾼다 |
| 태스크가 여섯 개면 | 모델 여섯 개 | 모델 여섯 개(또는 접두어로 한 모델) | 모델 하나 + 어댑터 하나 |

세 방식의 차이가 가장 잘 드러나는 곳이 **기계독해**입니다.
BERT(실습4A)는 지문에서 답의 **시작·끝 위치**를 고릅니다 — 지문에 없는 말은 원리적으로 낼 수 없습니다.
T5(실습4B)와 LLM은 답을 **써냅니다** — 그래서 지문에 없는 말도 쓸 수 있습니다. 편리한 만큼 지어낼 위험도 같이 옵니다.

오늘 쓰는 여섯 태스크는 이 성질을 기준으로 두 묶음입니다.

- **세 방식 모두 가능한 네 태스크** — 주제분류, 개체명인식, 기계독해, 문장유사도. §8에서 셋을 나란히 비교합니다.
- **생성 모델만 가능한 두 태스크** — SQL생성, 그리고 보너스인 수학추론. 정답이 라벨도 지문 속 구간도 아니라(새로 쓴 SQL 문, 계산 결과) **BERT 구조로는 만들 수가 없습니다.**

### 0-2. 어떤 모델로 할까 — 라인업 11종

1일차 강의에서 여러 계열의 언어모델을 소개했습니다. 그 가운데 **서로 구별되는 것만** 골라, 같은 데이터·같은 평가로
미리 돌려 두었습니다(`results/sweep2/`). 표가 답하려는 질문은 하나입니다 —
*"강의에서 들은 저 모델들이, 실제로 한국어 태스크를 학습시키면 이 정도 하는구나."*

| 묶음 | 모델 | 왜 이 모델인가 |
|---|---|---|
| **한국어 특화** | A.X-4.0-Light 7B (SKT) | 24GB GPU에서 학습되는 가장 큰 축. **수업에서 직접 학습**합니다 |
| | EXAONE-4.0-1.2B (LG) | 국내 대표 소형 모델. 라인업에서 가장 작아 크기 축의 아래쪽 |
| | kanana-1.5-2.1B **base / instruct** (Kakao) | 같은 몸에 후학습(지시 따르기)이 없고/있고 — *Base ↔ Instruct* 대조 |
| | Bllossom-3B (서울과기대) | Llama-3.2-3B에 **한국어를 추가학습**한 것 — 원본 Llama와 짝 |
| **글로벌** | Qwen3.5-2B (Alibaba) | 오픈웨이트 성능 선두 계열 |
| | Llama-3.2-3B-Instruct (Meta) | Bllossom의 원본. 한국어 추가학습 **전** 상태 |
| | Gemma-4-E2B-it (Google) | 5.1B로 적혀 있지만 임베딩 빼면 2B급 — 크기 표기의 함정 |
| | Ministral-3-3B (Mistral) | 미국·중국·한국 밖(유럽)의 대표 계열 |
| | gpt-oss-20B (OpenAI, MoE) | 크기 축의 끝. 21B 중 3.6B만 토큰마다 켜집니다 |

수업에서는 **A.X-4.0-Light (7.3B)** 를 씁니다. 강의장 GPU(RTX 4500 Ada 24GB)에서 학습되는 모델 중 큰 축이고,
**파인튜닝 뒤에도 수학추론이 떨어지지 않는** 모델입니다(§8에서 다시 봅니다). 시간이 모자라면 `MODEL_ID` 를
`kakaocorp/kanana-2-3b-instruct` 로 바꾸면 절반쯤 시간에 끝납니다.
아래 셀이 라인업의 학습 전·후 평균과, 한국어 특화 ↔ 글로벌 두 묶음의 평균을 보여 줍니다. 결과 파일이 아직 없는 모델은 "아직 없음"으로 나옵니다.

In [ ]:
import pandas as pd
from compare import load_sweep, LINEUP, LINEUP_GROUP, THREE_WAY_TASKS as _T4

def sweep_overview(sub="sweep2"):
    df = load_sweep(RESULTS, sub)
    df = df[df["tag"].isin(LINEUP)]                     # 에폭·4bit·plain 변형은 §5-1·§11에서 따로 본다
    if not len(df):
        return None, None
    g = (df.groupby(["tag", "model", "org", "kind"])
           .agg(**{"파라미터(B)": ("params_b", "first"), "학습(분)": ("train_min", "first"),
                   "GPU 최대(GB)": ("peak_gpu_gb", "first"),
                   "학습 전 평균(6)": ("before", "mean"), "학습 후 평균(6)": ("after", "mean")})
           .reset_index().round(1))
    g.insert(1, "묶음", g["tag"].map(LINEUP_GROUP))
    # LINEUP 순서로 정렬한다. set(g.index) 가 아니라 set(g["tag"]) 여야 한다 —
    # 이 시점의 g 는 reset_index() 뒤라 index 가 0,1,2… 이고 tag 는 열이다.
    have = set(g["tag"])
    g = g.set_index("tag").reindex([t for t in LINEUP if t in have]).reset_index()
    # 묶음별 평균 — 세 방식 공통 네 태스크만, 학습 전·후 둘 다 있는 모델만
    done = df.dropna(subset=["before", "after"])
    grp = (done[done["task"].isin(_T4)].assign(묶음=lambda d: d["tag"].map(LINEUP_GROUP))
             .groupby("묶음").agg(**{"모델 수": ("tag", "nunique"), "학습 전 평균(4)": ("before", "mean"),
                                    "학습 후 평균(4)": ("after", "mean")}).round(1))
    return g.rename(columns={"model": "모델", "org": "제작", "kind": "종류"}), grp

ov, grp = sweep_overview("sweep2")
if ov is None:
    print("results/sweep2/ 에 결과가 아직 없습니다 — 강사 스윕이 끝나면 이 표가 채워집니다.")
    print("수업 기본 모델은 skt/A.X-4.0-Light 입니다.")
else:
    shown = lambda t: t.astype(object).where(t.notna(), "아직 없음")   # 아직 안 끝난 칸은 빈칸 대신 글자로
    display(shown(ov.drop(columns="tag").set_index("모델")))
    print("\n묶음별 평균 — 세 방식 공통 네 태스크(주제분류·개체명인식·기계독해·문장유사도)")
    display(grp)

**읽는 법.** *Instruct*는 이미 "지시를 따르는 법"을 배운 모델, *Base*는 다음 단어 예측만 배운 모델입니다.
kanana의 base와 instruct를 비교해 보세요 — Base는 학습 전 점수가 낮지만 파인튜닝 후에는 같은 몸의 Instruct에 가까워집니다.
우리가 가르치는 것이 "지식"이 아니라 **형식과 태스크**라는 뜻입니다.
Llama-3.2-3B와 Bllossom-3B도 같은 식으로 읽습니다 — 한국어 추가학습이 준 출발점의 차이가 파인튜닝 뒤에도 남는지.
전체 비교(모델별 태스크별 점수, BERT·T5 기준선, 크기·비용)는 `bert_vs_llm.ipynb`에 있습니다.

### 0-3. 모델 카드 읽는 법 — 남이 만든 모델을 쓰기 전에 보는 것

우리가 쓸 모델은 **Hugging Face Hub** 에 올라와 있습니다. 저장소 주소 하나(`skt/A.X-4.0-Light`)만 있으면
`from_pretrained` 가 알아서 내려받습니다. 그 저장소의 첫 화면이 **모델 카드**(README)이고,
**모델을 쓰기 전에 여기부터 읽습니다.**

📄 [huggingface.co/skt/A.X-4.0-Light](https://huggingface.co/skt/A.X-4.0-Light) — 새 창으로 열어 두고 아래와 대조해 보세요.

| 카드에서 볼 것 | 왜 보나 | A.X-4.0-Light 의 경우 |
|---|---|---|
| **라이선스** | 수업·연구·상용에 쓸 수 있는가. 이것부터 본다 | **Apache-2.0** — 상용 포함 자유 |
| **접근 조건** | 그냥 받아지나, 약관 동의가 필요한가(gated) | 공개 — 동의 절차 없음 |
| **언어** | 한국어를 제대로 배운 모델인가 | `ko`, `en` |
| **크기** | 내 GPU 에 올라가나 | 7.26B (bf16 가중치만 약 15GB) |
| **문맥 길이** | 긴 지문을 넣을 수 있나 | 16,384 토큰 (우리 실습은 1,024면 충분) |
| **베이스 모델** | 무엇 위에 만들었나 — 토크나이저·대화 형식이 따라온다 | Qwen2.5 위에 한국어를 추가학습 |
| **사용 예시** | 어떤 형식으로 넣어야 하나 | `apply_chat_template` 로 대화를 문자열로 만든다 (§3 에서 그대로 씁니다) |
| **벤치마크** | 무엇을 잘한다고 주장하나. **우리 태스크와 다르면 참고만 한다** | KMMLU 64.2 · CLIcK 68.1 (같은 크기대 한국어 모델 중 상위) |
| **만든 곳·날짜** | 누가 책임지나, 얼마나 최신인가 | SKT AI Model Lab, 2025-07 |

> **벤치마크 숫자를 그대로 믿지 않습니다.** 카드의 점수는 만든 쪽이 고른 평가입니다.
> 우리가 이 노트북에서 하는 일이 바로 **내 태스크로 직접 재 보는 것**입니다 — §4 에서 학습 전 점수를 재고,
> §7 에서 학습 후와 비교합니다. 카드에서 좋아 보이던 모델이 내 태스크에서는 아닐 수 있습니다.

### 받아지지 않는 모델도 있습니다 — gated 와 토큰

일부 저장소는 **gated** 입니다. 웹에서 약관에 동의해야 내려받을 수 있고, 동의한 계정임을 증명하려면
**토큰**이 필요합니다(예: `naver-hyperclovax/HyperCLOVAX-SEED-Text-Instruct-1.5B`).
**이 수업의 모델은 전부 공개라 토큰이 필요 없습니다.** 다만 나중에 직접 쓸 때를 위해 절차만 적어 둡니다.

1. 모델 페이지에서 약관에 동의한다(승인이 필요한 곳도 있다)
2. Hugging Face 설정 → Access Tokens 에서 **읽기 전용(read)** 토큰을 만든다
3. 터미널에서 `hf auth login` 을 실행하고 토큰을 붙여 넣는다 — `~/.cache/huggingface/token` 에 저장된다

> ⚠️ **토큰을 코드나 노트북에 그대로 적지 마세요.** 저장소에 올라가면 남이 내 계정으로 쓸 수 있습니다.
> `hf auth login` 으로 한 번 저장하거나 환경변수 `HF_TOKEN` 을 쓰고, 노트북에는 **적지 않습니다.**

In [ ]:
from common import MAIN_TASKS, THREE_WAY_TASKS, GEN_ONLY_TASKS, TASKS

MODEL_ID = "skt/A.X-4.0-Light"                 # ← 다른 모델을 써 보려면 이 줄만 바꾸세요
# MODEL_ID = "kakaocorp/kanana-2-3b-instruct"   # 더 가볍게 (3.2B · GPU 8.8GB · 학습 13분)
# MODEL_ID = "LGAI-EXAONE/EXAONE-4.0-1.2B"      # 가장 가볍게 (1.2B · GPU 4.3GB · 학습 10분)

DATA_DIR = "data/llm-ft"                       # build_dataset.py 가 만드는 폴더
TRAIN_FILE = f"{DATA_DIR}/train_main.jsonl"    # 6태스크 학습셋 (주제분류·개체명·독해·유사도·수학·SQL)
LIMIT = 100                                    # 태스크별 평가 예제 수 (수업용. 강사 스윕은 300)
LIMIT_MATH = 100                               # 수학추론만 100건 — 아래 설명 참조
OUT_DIR = f"output/nb/{MODEL_ID.split('/')[-1]}"   # 어댑터·결과 저장 위치
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
print("모델      :", MODEL_ID)
print("학습 태스크:", MAIN_TASKS, "→", [TASKS[t]["name"] for t in MAIN_TASKS])
print("출력      :", OUT_DIR)

## 1. BERT식 vs LLM식 — 무엇이 다른가

1일차에 본 **전이학습(transfer learning)** 의 틀은 같습니다: 큰 말뭉치로 미리 학습한 모델(사전학습, pre-training)을 우리 데이터로 조금 더 학습(파인튜닝, fine-tuning)합니다.
다른 것은 *모델이 답을 내는 방식*입니다.

| | BERT식 (실습1·2·4A) | LLM식 (이번 실습) |
|---|---|---|
| 모델 | 인코더(양방향) — 문장을 읽고 벡터를 낸다 | 디코더(자기회귀) — 다음 토큰을 하나씩 **생성**한다 |
| 답을 내는 방법 | 태스크별 **head**(작은 분류기)를 새로 붙여 학습 | 답을 **글자로 쓴다** — head가 없다 |
| 태스크가 바뀌면 | head와 손실 함수, 전처리 코드를 바꾼다 | **지시문(prompt)** 만 바꾼다 |
| 태스크 여섯 개면 | 모델 여섯 개 | 모델 하나 + 어댑터 하나 |
| 라벨 | 정수 id (0, 1, 2 …) | 문자열 ("스포츠", "4.2", `[{"text":…}]`, `SELECT …`) |
| 학습 데이터 형식 | 태스크별로 다름 | 모두 **대화(messages)** 한 가지 |
| 파인튜닝 방식 | 전체 파라미터 학습(110M~340M) | **LoRA** — 1~2%만 학습 |
| 추론 | 한 번의 forward | 토큰 수만큼 반복 생성 → 느리다 |

용어 몇 개만 정리합니다.

- **Instruction tuning / SFT(Supervised Fine-Tuning)** — "지시 → 답" 쌍으로 LLM을 지도학습하는 것. ChatGPT류 모델을 만드는 첫 단계이기도 합니다 ([FLAN, Wei et al. 2022](https://arxiv.org/abs/2109.01652), [InstructGPT, Ouyang et al. 2022](https://arxiv.org/abs/2203.02155)).
- **Chat template** — "여기까지 사용자 말, 여기부터 답변"을 모델에게 알려주는 특수 토큰 규칙. 모델마다 다릅니다 (§3).
- **LoRA(Low-Rank Adaptation)** — 원래 가중치는 얼려 두고 작은 행렬 두 개만 학습하는 방법 ([Hu et al. 2021](https://arxiv.org/abs/2106.09685)) (§5).
- **어댑터(adapter)** — LoRA로 학습된 작은 가중치 파일. 원본 모델 + 어댑터 = 내 모델. 수십 MB면 충분합니다.

📚 더 읽기: ratsgo nlpbook [전이학습](https://ratsgo.github.io/nlpbook/docs/introduction/transfer/) · [언어모델](https://ratsgo.github.io/nlpbook/docs/language_model/) · Hugging Face LLM Course [Ch.11 Supervised Fine-Tuning](https://huggingface.co/learn/llm-course/chapter11/1)

### 이번 실습의 흐름

In [ ]:
# 그림은 SVG 로 그린다 — 글자로 그린 그림은 한글 폭 때문에 줄이 어긋난다
sys.path.insert(0, str(next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "tools" / "diagram.py").exists()) / "tools"))   # 앞 셀을 건너뛰어도 이 자리에서 다시 찾는다
from diagram import flow, show

show(flow([
    {"text": "원본 데이터", "sub": "KLUE · KorQuAD · GSM8K-ko · Spider-ko",
     "edge": "build_dataset.py 가 하나의 형식으로 바꾼다", "tag": "§2"},
    {"text": "train_main.jsonl", "sub": "6태스크 4,800건 · messages 형식", "mono": True,
     "edge": "tokenizer + chat template", "tag": "§3"},
    {"text": "학습 전 평가", "sub": "기준선 — 아직 아무것도 가르치지 않은 상태", "kind": "eval",
     "edge": "LoRA 어댑터를 붙이고 1 epoch", "tag": "§4"},
    {"text": "학습", "sub": "PEFT + TRL · 전체 파라미터의 0.55%만", "kind": "accent",
     "edge": "같은 평가셋 · 같은 채점 함수", "tag": "§5·§6"},
    {"text": "학습 후 평가", "sub": "전후 비교 → BERT·T5와 비교", "kind": "eval", "tag": "§7·§8"},
    {"text": "데모 페이지", "sub": "내가 학습시킨 모델에게 직접 물어보기", "kind": "out", "tag": "§9"},
]))

## 2. 데이터 — 여섯 태스크를 하나의 형식으로

| 코드 | 태스크 | 데이터셋 | 입력 | 모델이 써야 하는 답 | 지표 | BERT로도 되나 |
|---|---|---|---|---|---|---|
| `tc` | 주제분류 | [KLUE-YNAT](https://klue-benchmark.com/tasks/66/overview/description) | 뉴스 제목 | 7개 주제 중 하나 | 정확도 | 된다 |
| `ner` | 개체명인식 | [KLUE-NER](https://klue-benchmark.com/tasks/69/overview/description) | 문장 | `[{"text": "…", "label": "PS"}]` | F1 | 된다 |
| `mrc` | 기계독해 | [KorQuAD 1.0](https://korquad.github.io/KorQuad%201.0/) | 지문 + 질문 | 지문 속 답 구간 | EM | 된다 |
| `sts` | 문장유사도 | [KLUE-STS](https://klue-benchmark.com/tasks/67/overview/description) | 문장 두 개 | 0.0 ~ 5.0 점수 | Pearson | 된다 |
| `math` | 수학추론 | [GSM8K-ko](https://huggingface.co/datasets/kuotient/gsm8k-ko) | 문장제 문제 | 풀이 + `#### 답` | EM | **안 된다** |
| `sql` | SQL생성 | [Spider-ko](https://huggingface.co/datasets/huggingface-KREW/spider-ko) | DB 스키마 + 질문 | SQL 문 한 줄 | EM | **안 된다** |

앞의 네 개는 1일차 BERT 실습과 **같은 평가셋**이라 바로 비교할 수 있습니다.
뒤의 두 개는 정답이 라벨도 지문 속 구간도 아니라서 BERT 구조로는 만들 수가 없습니다 — 생성 모델만 답할 수 있는 자리입니다.

### 왜 감성분류(NSMC)와 자연어추론(KLUE-NLI)을 뺐나

작년까지 쓰던 여섯 태스크에는 감성분류와 자연어추론이 들어 있었습니다. 이번에 둘을 빼고 SQL생성·수학추론을 넣었습니다.

- **감성분류(NSMC)** — 요즘 모델은 학습 전에도 이미 잘합니다. 학습 전과 후가 거의 같아서 "파인튜닝이 무엇을 바꾸는가"를 보여 주지 못합니다.
- **자연어추론(KLUE-NLI)** — 문장유사도와 자리가 겹칩니다(둘 다 문장 쌍을 받아 관계를 판정). 둘 중 하나만 남긴다면 문장유사도가 낫습니다. 이유는 두 가지입니다.
  1. **설명하기 쉽다.** "두 문장이 얼마나 비슷한가를 0~5점으로" 는 배경 지식 없이 바로 이해됩니다. 반면 "함의 / 중립 / 모순"은 개념 자체를 먼저 설명해야 합니다.
  2. **파인튜닝 효과가 훨씬 극적이다.** 학습이 끝난 뒤 점수는 두 태스크가 사실상 같지만, **학습 전→후의 변화 폭**은 문장유사도가 압도적으로 큽니다. 아래 셀에서 실제 숫자로 확인합니다.

아래 셀은 **이전 구성(감성분류·자연어추론이 있던 6태스크) 스윕 결과**를 읽어 그 근거를 보여 줍니다. 새 표(§8)와 섞어 읽지 마세요.

In [ ]:
# [이전 구성 기준] 문장유사도 vs 자연어추론 — 학습 전/후 변화 폭 비교 (results/sweep/)
from compare import TASK_NAME, MAIN_METRIC

def read_results(sub, tag, kind):
    p = Path(RESULTS) / sub / f"{tag}-{kind}.json"
    return json.loads(p.read_text(encoding="utf-8")).get("results", {}) if p.exists() else {}

rows = []
for meta in sorted((Path(RESULTS) / "sweep").glob("*-train_meta.json")):
    tag = meta.name[: -len("-train_meta.json")]
    b, a = read_results("sweep", tag, "before"), read_results("sweep", tag, "after")
    for t in ["sts", "nli"]:
        m = MAIN_METRIC[t]
        if t in b or t in a:
            rows.append({"모델": tag, "태스크": TASK_NAME[t],
                         "학습 전": (b.get(t) or {}).get(m), "학습 후": (a.get(t) or {}).get(m)})
if rows:
    df = pd.DataFrame(rows)
    df["변화"] = df["학습 후"] - df["학습 전"]
    piv = df.pivot_table(index="모델", columns="태스크", values=["학습 전", "학습 후", "변화"])
    display(piv.reindex(columns=["학습 전", "학습 후", "변화"], level=0).round(1))
    print("\n평균:")
    display(df.groupby("태스크")[["학습 전", "학습 후", "변화"]].mean().round(1))
else:
    print("results/sweep/ (이전 구성) 결과가 없어 이 비교는 건너뜁니다.")

> 학습 **후** 점수는 두 태스크가 비슷하지만, **학습 전** 점수와 변화 폭은 문장유사도가 훨씬 큽니다.
> 문장유사도는 지시문만 보고 "0~5 사이 소수"를 내놓는 일이라 학습 전 모델이 형식부터 못 맞추고, 심하면 상관이 **음수**로 나옵니다.
> 그래서 "파인튜닝이 무엇을 해 주는가"를 한 장면으로 보여 주기에 좋습니다.

**평가셋 분리 원칙.** 학습셋은 각 데이터셋의 공식 `train`에서만, 평가셋은 공식 `test`/`validation`에서만 만듭니다.
스크립트가 두 집합의 문장 중복을 검사해 **겹치는 것은 학습셋에서 뺍니다**. 평가셋으로 학습하면 점수는 오르지만 아무 의미가 없습니다.
SQL생성은 학습용 데이터베이스와 평가용 데이터베이스가 아예 겹치지 않아, 리키지가 코드가 아니라 **데이터 구조로** 막혀 있습니다.

데이터 라이선스: KLUE CC BY-SA 4.0 · KorQuAD CC BY-ND 2.0 KR · GSM8K MIT · Spider CC BY-SA 4.0.
(강의 자료는 저작권법 제25조·제37조에 따라 출처를 표시하고 교육 목적으로 이용합니다.)

In [ ]:
# 데이터가 없으면 만든다 (원본 data/ 폴더에서 읽어 data/llm-ft/ 에 씀.
#  KLUE·GSM8K-ko·Spider-ko 원본이 없으면 Hugging Face에서 자동으로 내려받는다)
missing = [t for t in MAIN_TASKS if not Path(f"{DATA_DIR}/eval_{t}.jsonl").exists()]
if not Path(TRAIN_FILE).exists() or missing:
    print("없는 것:", ("학습셋 " if not Path(TRAIN_FILE).exists() else "") + (f"평가셋 {missing}" if missing else ""))
    !{PY} task5-llm-ft/build_dataset.py
else:
    print("이미 있음:", TRAIN_FILE)

In [ ]:
from common import SYSTEM_PROMPT, read_jsonl

stats = json.loads(Path(f"{DATA_DIR}/stats.json").read_text(encoding="utf-8"))
rows = [{"코드": t, "태스크": v["name"], "데이터셋": v["dataset"], "학습": v["train"], "평가": v["eval"],
         "중복 제거": v["dropped_by_leak"], "학습 원본": v["train_src"], "평가 원본": v["eval_src"]}
        for t, v in stats["per_task"].items() if t in MAIN_TASKS]
display(pd.DataFrame(rows).set_index("코드"))

train_rows = read_jsonl(TRAIN_FILE)
counts = pd.Series([r["task"] for r in train_rows]).value_counts().reindex(MAIN_TASKS).to_dict()
print(f"학습셋 합계: {len(train_rows)}건 ({TRAIN_FILE})")
print("태스크별   :", counts)

학습 파일의 한 줄은 **대화(messages)** 하나입니다 — `system`(역할 설명) → `user`(지시문 + 입력) → `assistant`(정답).
여섯 태스크가 모두 이 한 형식입니다. 태스크를 바꾸는 것은 `user` 안의 **지시문**뿐입니다.

### 원본 한 줄 → 변환된 학습 예제

`build_dataset.py` 가 원본 데이터셋(형식이 저마다 다름)을 어떻게 **하나의 대화**로 바꾸는지,
태스크마다 실물 1건으로 봅니다. 표로 자르지 않고 전문을 보여 줍니다 — 특히 개체명·기계독해는
`raw`(BERT 기준선용 원본 정답 형식)와 `messages`(LLM용 대화)가 같은 정답을 어떻게 다르게
표현하는지 눈여겨보세요.

In [ ]:
from common import raw_sample, raw_to_train_html
from IPython.display import HTML, display

seen = set()
for r in train_rows:
    if r["task"] in seen:
        continue
    seen.add(r["task"])
    raw_row = raw_sample(r["task"], 1)[0]
    display(HTML(raw_to_train_html(r["task"], raw_row, r)))

In [ ]:
# 지시문 전문 — 모델이 보는 '문제지'. 태스크를 바꾸는 유일한 부분이다.
print("[system]", SYSTEM_PROMPT, "\n")
for t in ["tc", "sts", "math", "sql"]:
    print(f"── {t} ({TASKS[t]['name']}) 지시문 ──")
    print(TASKS[t]["instruction"])
    print()

> **관찰 포인트 ①** 지시문은 "무엇을 하라"와 함께 **출력 형식**(7개 라벨 중 하나만, 소수점 한 자리, `#### 답` 줄, 코드블록 표시 없이 SQL 한 줄)을 명시합니다.
> LLM에게 라벨 집합은 코드가 아니라 **문장**으로 전달됩니다. 형식을 안 지키면 채점기가 "형식 깨짐"으로 셉니다 — 학습 전 모델이 얼마나 자주 그러는지 §4에서 봅니다.

## 3. 토크나이저와 chat template

모델은 글자를 직접 읽지 못하고 **토큰(token)** 이라는 조각의 번호를 읽습니다(1일차 복습). BERT는 WordPiece(`##` 접두), 요즘 LLM은 대개 BPE 계열이며 어휘 크기가 10만~25만으로 훨씬 큽니다.

그리고 LLM에는 한 가지가 더 있습니다. "여기부터 사용자, 여기부터 답변"을 구분하는 **chat template** 입니다.
`messages` 리스트가 이 템플릿을 거쳐 **하나의 긴 문자열**이 되고, 그것이 토큰으로 바뀌어 모델에 들어갑니다. 템플릿은 모델마다 다릅니다:

| 모델 | 사용자 턴 시작 | 답변 종료 토큰 |
|---|---|---|
| Qwen | `<\|im_start\|>user` | `<\|im_end\|>` |
| EXAONE-4.0 | `[\|user\|]` | `[\|endofturn\|]` |
| Llama-3 계열(kanana, Llama-3.2, Bllossom) | `<\|start_header_id\|>user<\|end_header_id\|>` | `<\|eot_id\|>` |
| Gemma-4 | `<\|turn>user` | `<turn\|>` |
| **Base 모델** | (없음 → 우리가 `[사용자]` / `[답변]` 단순 형식을 붙임) | `eos_token` |

**종료 토큰**이 중요합니다. 모델은 답을 쓴 뒤 이 토큰을 내야 멈춥니다. 학습 데이터에 종료 토큰이 들어 있어야 "답하고 멈추는 것"까지 배웁니다 (§10 사례 연구).

📚 [Hugging Face — Chat Templates](https://huggingface.co/docs/transformers/chat_templating) · [LLM Course Ch.2 토크나이저](https://huggingface.co/learn/llm-course/chapter2/4) · [Tokenizer Playground](https://huggingface.co/spaces/Xenova/the-tokenizer-playground)

In [ ]:
from transformers import AutoTokenizer
from common import ensure_chat_template, build_prompt, render_example, make_messages, user_message

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
template_kind = ensure_chat_template(tokenizer)      # Base 모델이면 단순 형식을 붙인다
print(f"어휘 크기 {len(tokenizer):,} | chat template: {'모델 고유' if template_kind == 'native' else '없음 → 단순 형식 부착'}")
print(f"eos_token = {tokenizer.eos_token!r}   pad_token = {tokenizer.pad_token!r}")

# BERT 토크나이저와 비교해 보자 (실습1에서 쓴 klue/roberta-base)
bert_tok = AutoTokenizer.from_pretrained("klue/roberta-base")
sent = "ETRI 언어지능연구실에서 2026년 9월 3일 자연어처리 실습을 진행했다."
print("\n문장:", sent)
print(f"LLM  토큰 {len(tokenizer.tokenize(sent)):2d}개:", tokenizer.tokenize(sent))
print(f"BERT 토큰 {len(bert_tok.tokenize(sent)):2d}개:", bert_tok.tokenize(sent))

In [ ]:
# 학습에 들어가는 텍스트를 '그대로' 본다 — 대화 형식이 템플릿을 거쳐 한 문자열이 된 모습
sample = train_rows[0]["messages"]
rendered = render_example(tokenizer, sample)
print("─── 학습 텍스트 (앞 900자) ───")
print(rendered[:900])
print("\n─── 추론 프롬프트의 끝 (답변 자리까지만) ───")
infer_prompt = build_prompt(tokenizer, sample)
print(repr(infer_prompt[-120:]))
print("\n학습 텍스트가 추론 프롬프트로 시작하는가:", "예 (일관됨)" if rendered.startswith(infer_prompt) else "아니오 — 확인 필요!")
ids = tokenizer(rendered, add_special_tokens=False)["input_ids"]
print(f"토큰 수 {len(ids)} | 마지막 토큰 3개: {tokenizer.convert_ids_to_tokens(ids[-3:])}")

> **관찰 포인트 ②** 학습 텍스트의 **마지막 토큰**이 무엇인지 보세요. 그것이 이 모델의 "답변 종료 토큰"입니다. 추론 프롬프트는 답변 자리 직전까지만 있고, 그 뒤를 모델이 이어 씁니다.
> EXAONE·Qwen처럼 *thinking 모드*가 있는 모델은 `enable_thinking=False`로 고정해 생각 블록 없이 답만 바로 내게 합니다(`common.build_prompt`).

### 3-1. 멀티턴 — 대화가 여러 번 오갈 때

우리 학습 데이터는 한 번 묻고 한 번 답하는 **한 턴**짜리입니다. 하지만 실제 대화형 모델을 학습시킬 때는
`user` → `assistant` → `user` → `assistant` … 가 **여러 번 반복되는** 대화 하나가 학습 예제 하나가 됩니다.
형식은 똑같습니다 — `messages` 리스트가 길어질 뿐입니다.

아래 셀은 오늘 태스크 넷을 이어 붙여 **4턴짜리 대화 한 건**을 직접 만들고, 그것이 chat template을 거쳐 어떤 문자열이 되는지 봅니다.
두 가지를 확인하세요.

1. **종료 토큰이 턴마다 반복된다.** 답변이 네 번이면 종료 토큰도 네 번 나옵니다. 모델은 "여기서 멈춘다"를 네 번 배웁니다.
2. **`assistant_only_loss`는 여러 답변 구간에 걸린다.** 손실을 답변 부분에만 걸면, 한 줄(=대화 한 건) 안에서 **서로 떨어진 네 구간**에만 손실이 걸리고 지시문 구간은 전부 빠집니다.

In [ ]:
# 4턴 합성 대화 — 여기서 직접 만든다 (외부 데이터셋이 아니다)
MULTITURN = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": user_message("tc", {"text": "손흥민 결승골… 토트넘, 원정서 2-1 역전승"})},
    {"role": "assistant", "content": "스포츠"},
    {"role": "user", "content": user_message("sts", {"sentence1": "오늘 날씨가 정말 좋다.",
                                                     "sentence2": "날씨가 참 맑고 화창하네."})},
    {"role": "assistant", "content": "4.2"},
    {"role": "user", "content": user_message("ner", {"text": "김연아는 2010년 밴쿠버 올림픽에서 금메달을 땄다."})},
    {"role": "assistant", "content": '[{"text": "김연아", "label": "PS"}, {"text": "2010년", "label": "DT"}, {"text": "밴쿠버", "label": "LC"}]'},
    {"role": "user", "content": user_message("math", {"question": "연필 한 자루는 700원, 공책 한 권은 1200원이다. 연필 3자루와 공책 2권을 사면 모두 얼마인가?"})},
    {"role": "assistant", "content": "연필 3자루는 700 x 3 = 2100원이다.\n공책 2권은 1200 x 2 = 2400원이다.\n합하면 2100 + 2400 = 4500원이다.\n#### 4500"},
]

from common import chat_html
display(HTML(chat_html(MULTITURN, title="4턴 대화 — 역할별로 색을 나눠 봅니다")))

mt_text = render_example(tokenizer, MULTITURN)
print("── 실제로 모델에 들어가는 최종 문자열(참고용, 특수 토큰 포함) ──")
print(mt_text)

In [ ]:
from common import end_of_turn_token_id

# (1) 종료 토큰이 턴마다 반복되는가
eid = end_of_turn_token_id(tokenizer)
eot = tokenizer.decode([eid]) if eid is not None else (tokenizer.eos_token or "")
answers = [m["content"] for m in MULTITURN if m["role"] == "assistant"]
print(f"답변 턴 수      : {len(answers)}")
print(f"종료 토큰       : {eot!r}")
print(f"텍스트에 등장   : {mt_text.count(eot) if eot else 0}회")
print(f"전체 토큰 수    : {len(tokenizer(mt_text, add_special_tokens=False)['input_ids'])}")

# (2) 손실이 걸리는 구간 — 답변 문자열이 있는 자리만
print("\n답변 부분에만 손실(assistant_only_loss)을 걸면 손실이 걸리는 구간:")
pos, covered = 0, 0
for i, a in enumerate(answers, 1):
    j = mt_text.find(a, pos)
    if j < 0:
        print(f"  {i}번째 답변: 텍스트에서 찾지 못함 (템플릿이 내용을 바꿔 쓰는 모델)")
        continue
    covered += len(a)
    print(f"  {i}번째 답변 → 글자 {j:5d} ~ {j+len(a):5d}   {a.splitlines()[0][:40]!r}")
    pos = j + len(a)
print(f"\n답변이 차지하는 글자 비율: {covered/len(mt_text)*100:.1f}%  "
      f"(나머지 {100-covered/len(mt_text)*100:.1f}%는 지시문 — 여기에는 손실을 걸지 않는다)")

> **관찰 포인트 ②-1** 지시문이 텍스트의 대부분을 차지합니다. 손실을 전체 시퀀스에 걸면 모델은 **자기가 받은 문제지를 외우는 데** 학습량의 상당 부분을 씁니다.
> 답변 구간에만 걸면 그 몫이 전부 정답 쓰기로 갑니다. 이 차이가 실제 점수에 얼마나 나타나는지는 §6에서 다시 봅니다.
> (한 가지 주의: 두 방식의 **손실 값은 서로 비교할 수 없습니다** — 답변 토큰만 세면 분모가 작아져 손실이 훨씬 작게 나옵니다.)

### 3-2. 학습 텍스트가 조용히 망가지는 세 가지

파인튜닝이 실패하는 원인은 대개 학습률이나 LoRA rank가 아니라 **학습 텍스트**입니다.
그리고 이 사고들은 오류 메시지를 내지 않습니다 — 학습은 끝까지 돌고, 점수만 이상하게 나옵니다.
자주 나오는 셋을 직접 재현해 봅니다.

**(a) 종료 토큰을 특수 토큰이 아니라 문자열로 붙이기.**
`answer + "</s>"` 처럼 다른 모델에서 베껴 온 문자열을 정답 뒤에 그냥 이어 붙이는 경우입니다.
그 모델의 어휘에 없는 문자열이면 **평범한 글자 여러 개**로 쪼개져 들어갑니다.
모델은 "멈춰라"를 배우는 대신 **`<`, `/`, `s`, `>` 를 차례로 쓰는 법**을 배웁니다. 그리고 여전히 멈추지 않습니다.
종료 토큰은 chat template이 넣게 두는 것이 안전합니다.

**(b) 프롬프트 템플릿에 주석·들여쓰기가 섞여 모든 예제 앞에 붙기.**
지시문을 함수 안에서 여러 줄 문자열(삼중 따옴표)로 쓰다 보면 맨 앞의 줄바꿈, 들여쓰기 공백, 설명용 `#` 주석이 문자열에 그대로 들어갑니다.
사람 눈에는 안 보이지만 **모든 학습 예제 앞에 똑같이** 붙고, 예제 수만큼 곱해집니다.
더 나쁜 것은 추론할 때 그 앞머리를 빼먹는 경우입니다 — 그러면 (c)가 됩니다.

**(c) 학습 프롬프트와 추론 프롬프트가 어긋나기.**
학습은 시스템 프롬프트 A로, 추론은 B로 하면 모델은 **본 적 없는 문맥**에서 답하게 됩니다. 점수만 조용히 떨어집니다.
`train.py`는 학습을 시작하기 전에 이것을 **자동으로 점검**합니다: 같은 예제로 학습 텍스트와 추론 프롬프트를 각각 만들어
`rendered.startswith(infer_prompt)` 를 확인하고 `학습 텍스트가 추론 프롬프트로 시작하는가: 예/아니오` 를 찍습니다.
"아니오"가 나오면 그 자리에서 멈추고 원인을 찾아야 합니다. 노트북에서는 §3의 셀이 같은 점검을 했습니다.

In [ ]:
ex_tc = {"text": "손흥민 결승골… 토트넘, 원정서 2-1 역전승"}

# ── (a) 종료 토큰: 진짜 특수 토큰 vs 베껴 온 문자열 ───────────────────────────────
print("(a) 종료 토큰을 문자열로 붙이면")
for s in [eot, "</s>", "<|im_end|>"]:
    if not s:
        continue
    tid = tokenizer(s, add_special_tokens=False)["input_ids"]
    kind = "특수 토큰 1개" if len(tid) == 1 else f"평범한 글자 {len(tid)}개로 쪼개짐"
    print(f"   {s!r:16s} → {kind:22s} {tokenizer.convert_ids_to_tokens(tid)}")
print("   ↑ 쪼개진 쪽을 정답 뒤에 붙이면 모델은 '멈추기'가 아니라 '그 글자들 쓰기'를 배운다.\n")

# ── (b) 앞에 붙은 보이지 않는 글자 ──────────────────────────────────────────────
def bad_instruction(text):
    # 함수 안에서 삼중따옴표로 쓰면 줄바꿈·들여쓰기·주석이 문자열에 그대로 들어간다
    return """
    # 뉴스 제목의 주제를 분류한다 (라벨 7개)
    다음 뉴스 제목의 주제를 분류하세요.
    제목: {text}""".format(text=text)

good, bad = user_message("tc", ex_tc), bad_instruction(ex_tc["text"])
n_good = len(tokenizer(good, add_special_tokens=False)["input_ids"])
n_bad = len(tokenizer(bad, add_special_tokens=False)["input_ids"])
print("(b) 템플릿에 섞여 들어간 주석·들여쓰기")
print(f"   제대로 쓴 것 앞 40자: {good[:40]!r}")
print(f"   망가진 것  앞 40자: {bad[:40]!r}")
print(f"   토큰 수 {n_good} → {n_bad} (+{n_bad-n_good}). 학습 예제 {len(train_rows)}건이면 "
      f"쓸데없는 토큰 {(n_bad-n_good)*len(train_rows):,}개가 더 들어간다.\n")

# ── (c) 학습 프롬프트와 추론 프롬프트 어긋남 ────────────────────────────────────
train_text = render_example(tokenizer, make_messages("tc", ex_tc, "스포츠"))
infer_ok = build_prompt(tokenizer, make_messages("tc", ex_tc))
infer_bad = build_prompt(tokenizer, [{"role": "system", "content": "당신은 친절한 AI 비서입니다."},
                                     {"role": "user", "content": user_message("tc", ex_tc)}])
print("(c) 학습 텍스트가 추론 프롬프트로 시작하는가  ← train.py가 학습 전에 자동으로 찍는 줄")
print(f"   같은 시스템 프롬프트 : {'예 (일관됨)' if train_text.startswith(infer_ok) else '아니오'}")
print(f"   다른 시스템 프롬프트 : {'예 (일관됨)' if train_text.startswith(infer_bad) else '아니오 — 확인 필요!'}")
i = next((k for k in range(min(len(train_text), len(infer_bad))) if train_text[k] != infer_bad[k]), None)
if i is not None:
    print(f"   처음 어긋나는 자리 {i}: 학습 {train_text[i:i+30]!r} / 추론 {infer_bad[i:i+30]!r}")

## 4. 학습 전 모델은 얼마나 할까 — 기준선

파인튜닝 효과를 말하려면 **같은 평가셋으로 학습 전 점수**를 먼저 재야 합니다. 지표는 1일차 실습과 같고, 쉬운 말로는:

- **정확도(accuracy)** — 맞힌 비율. (tc)
- **F1** — 찾아낸 개체명 중 맞은 비율(정밀도)과 실제 개체명 중 찾아낸 비율(재현율)의 조화평균. (ner)
- **EM(exact match)** — 답이 정답과 글자까지 똑같은 비율. (mrc, math, sql. mrc는 부분 점수 F1도 함께 봅니다)
- **Pearson 상관** — 모델 점수와 사람 점수가 같이 오르내리는 정도, −1~1을 여기서는 ×100. (sts)
- **형식 깨짐(broken_format)** — 답이 지정 형식이 아니어서 채점할 수 없었던 건수. LLM 특유의 지표입니다.

모델을 GPU에 올립니다(bf16). 이 모델 객체를 §5~§7에서 계속 씁니다.

In [ ]:
from transformers import AutoModelForCausalLM
from common import check_end_token

t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16, device_map="cuda")
end_info = check_end_token(model, tokenizer, MODEL_ID)     # 종료 토큰을 모델이 낼 수 있는지 점검 (§10)
n_params = sum(p.numel() for p in model.parameters())
print(f"로드 {time.time()-t0:.0f}초 | 파라미터 {n_params/1e9:.2f}B | GPU 사용 {torch.cuda.memory_allocated()/1024**3:.1f} GB")
print("종료 토큰 점검:", end_info)

### 미션 `m-eval` — 평가 루프의 첫 단계

점수를 내려면 먼저 **모델에게 제대로 물어야** 합니다. 그 "묻는" 부분을 직접 만듭니다.

평가는 세 걸음입니다.

| | 하는 일 | 누가 |
|---|---|---|
| ① | 태스크 지시문 + 입력 → 모델이 읽을 문자열 | **여러분** |
| ② | 그 문자열로 답을 생성 | **여러분** |
| ③ | 답을 정답과 맞춰 채점 | `common.py` 의 채점 함수 |

③은 이미 만들어져 있습니다(`pick_label`·`parse_score`·`score_ner` …). 여러분이 채울 것은 `ask()` 안의 **`____` 두 곳**입니다.

> **왜 이것이 미션인가.** 학습할 때 모델이 본 문장과 물어볼 때 주는 문장이 **한 글자라도 다르면**
> 모델은 제 실력을 내지 못합니다. §3-2에서 본 "조용히 망가지는 세 가지"가 바로 그 이야기였습니다.
> 그래서 이 저장소는 지시문을 `common.py` **한 곳에만** 두고 학습·평가·데모가 모두 같은 함수를 부릅니다.
> 확인 셀에서 그 둘이 정말 같은지 글자 단위로 맞춰 봅니다.

아래 객관식 두 개를 먼저 풀어 보세요. 고른 답을 그 다음 셀의 `____` 에 그대로 옮기면 됩니다.
막히면 **힌트**를 단계별로 열 수 있습니다.

In [ ]:
quizkit.show("m-eval-1")

In [ ]:
quizkit.show("m-eval-2")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  미션 m-eval-1 · m-eval-2 — 지시문을 만들어 모델에게 묻기
# ══════════════════════════════════════════════════════════════════════
#
# 참고. 같은 일을 하는 코드는 `task5-llm-ft/serve.py` 에 있습니다 — 감추지 않고 알려 드립니다.

from common import generate_one, TASK_MAX_TOKENS

# 아래 ____ 두 곳만 채우세요. 바로 위 셀의 객관식에서 고른 답을 그대로 옮기면 됩니다.

def ask(task, m=None, **payload):
    "지시문을 만들어 모델에게 묻는다. 데모 페이지(serve.py)가 하는 일과 같다."
    # Step 1·2. 태스크 지시문 + 입력 -> 대화 메시지 -> chat template 을 적용한 한 덩어리 문자열
    prompt = build_prompt(tokenizer, ____)                                  # m-eval-1
    # Step 3. 그 문자열로 답을 생성한다 (태스크마다 최대 생성 길이가 다르다)
    return generate_one(m if m is not None else model, tokenizer, prompt,
                        ____)                                               # m-eval-2

#### 확인

> 빈칸을 채우지 않으면 여기서 `NameError: name '____' is not defined` 로 멈춥니다. 정상입니다 —
> 위 셀의 `____` 두 곳을 채우고 그 셀을 다시 실행한 뒤 이 셀을 실행하세요.

In [ ]:
# 학습 데이터에 든 지시문과, 방금 만든 ask() 가 만드는 지시문이 글자까지 같은가?
_row = read_jsonl(f"{DATA_DIR}/eval_tc.jsonl", 1)[0]
_msgs = make_messages("tc", _row["input"])

assert _msgs[-1]["role"] == "user", "마지막 메시지는 사용자 메시지여야 합니다"
assert _msgs[-1]["content"] == _row["messages"][1]["content"], (
    "학습 데이터의 지시문과 지금 만든 지시문이 다릅니다 — 이러면 모델이 제 실력을 못 냅니다")

_prompt = build_prompt(tokenizer, _msgs)
assert _row["messages"][1]["content"] in _prompt, (
    "chat template 을 적용한 문자열 안에 지시문이 그대로 들어 있어야 합니다")

_out = ask("tc", **_row["input"])
assert isinstance(_out, str), f"ask() 는 문자열을 돌려줘야 합니다 (지금 {type(_out).__name__})"

print("통과 — 학습 때 쓴 지시문과 물어볼 때 만든 지시문이 글자까지 같습니다.")
print(f"  물어본 것 : {_row['input']['text']}")
print(f"  정답      : {_row['gold']}")
print(f"  모델 답   : {_out!r}   (아직 학습 전이라 엉뚱해도 정상입니다)")

이제 여섯 태스크에 하나씩 물어봅니다. **학습 전** 모델이라 엉뚱하거나 형식이 깨진 답이 나오는 것이 정상입니다 — 그것이 기준선입니다.

In [ ]:
DEMO_SCHEMA = ("student : student_id (number) , name (text) , dept_id (number) , gpa (number) | "
               "department : dept_id (number) , dept_name (text)")

EXAMPLES = [
    ("tc",   dict(text="손흥민 결승골… 토트넘, 원정서 2-1 역전승")),
    ("ner",  dict(text="김연아는 2010년 밴쿠버 올림픽에서 금메달을 땄다.")),
    ("mrc",  dict(context="한국전자통신연구원(ETRI)은 1976년 설립된 정부출연연구기관으로 대전에 본원을 두고 있다.",
                  question="ETRI 본원은 어디에 있나?")),
    ("sts",  dict(sentence1="오늘 날씨가 정말 좋다.", sentence2="날씨가 참 맑고 화창하네.")),
    ("math", dict(question="연필 한 자루는 700원, 공책 한 권은 1200원이다. 연필 3자루와 공책 2권을 사면 모두 얼마인가?")),
    ("sql",  dict(schema=DEMO_SCHEMA, question="학점이 3.5 이상인 학생의 이름을 알려줘.")),
]
model.eval()
for task, payload in EXAMPLES:
    print(f"[{task}] → {ask(task, **payload)!r}\n")

학습 전 모델의 답을 보았으니, 이제 평가셋으로 점수를 냅니다. `evaluate_tasks`는 태스크마다 정해진 건수를 생성하고 채점합니다.
수학추론은 답에 풀이 과정이 들어가 **생성이 길고 느립니다.** 그래서 건수가 곧 기다리는 시간입니다.

여기서는 여섯 태스크를 모두 **100건**으로 봅니다. 수업 중에 "그렇구나" 하고 넘어갈 만한 크기입니다.
다만 수학추론은 **한 문제를 맞히고 틀리는 것이 점수를 크게 흔드는** 태스크라는 것을 알고 보아야 합니다 —
100건이면 한 문제가 1점입니다. 학습 전후 차이가 몇 점 나더라도 그것이 실제 변화인지
표본 탓인지 이 크기로는 단정할 수 없습니다.

**강사가 태스크당 300건으로 미리 재 둔 값이 `docs/LECTURE-FACTS.md` 에 있습니다.**
수업에서 결론을 말할 때는 그 표를 씁니다. 여기서 보는 것은 "내 화면에서도 같은 방향이 나오는가"입니다.

**얼마나 흔들리는지 실제로 재 봤습니다.** 강사가 이 모델의 학습 전·후 답 300건을 받아 두고,
앞에서부터 몇 건까지 채점하느냐만 바꿔 본 것입니다. **답은 하나도 다시 만들지 않았습니다** —
같은 예측을 몇 개까지 세느냐만 다릅니다.

| 채점한 건수 | 학습 전 | 학습 후 | 차이 |
|---:|---:|---:|---:|
| 50 | 52.0 | 40.0 | **−12.0** |
| 100 | 50.0 | 47.0 | −3.0 |
| 150 | 48.0 | 48.0 | 0.0 |
| 200 | 50.0 | 51.0 | +1.0 |
| 300 | 50.7 | 53.7 | **+3.0** |

50건에서는 **12점 떨어졌다**고 읽히고, 300건에서는 **3점 올랐다**고 읽힙니다. 같은 모델의 같은 답인데도 그렇습니다.

100건이 특별히 안전한 것도 아닙니다. 300건 중 **어느 100건을 고르느냐**만 바꾸면 학습 후 점수가
47 · 52 · 55 · 58 · 59 로 나옵니다 — 같은 모델에서 12점이 왔다 갔다 합니다.

> **적게 재면 없는 변화가 보입니다.** 숫자 하나를 그대로 믿기 전에 몇 건으로 잰 것인지 보는 습관이
> 이 실습에서 가져갈 것 하나입니다. 위 표는 §11 「더 해보기」 1번으로 직접 재현할 수 있습니다.
터미널에서는 `python task5-llm-ft/evaluate.py --tasks tc,ner,mrc,sts,math,sql --limit 100 --show 2` 가 같은 일을 합니다.

### 퀴즈 `q-llm-3` — 같은 답인데 결론이 뒤집힌 까닭

위 표에서 무엇을 결론으로 가져가야 하는지 묻는 문제입니다. 보기를 고르면 해설이 열립니다.

In [ ]:
quizkit.show("q-llm-3")

In [ ]:
from common import evaluate_tasks, results_table

EVAL_TASKS = [t for t in MAIN_TASKS if Path(f"{DATA_DIR}/eval_{t}.jsonl").exists()]
print("평가할 태스크:", EVAL_TASKS)

# 평가는 학습보다 GPU 를 더 씁니다 — 한 번에 여러 예제를 생성하기 때문입니다.
# 24GB 카드에서 7.3B 모델을 쓰므로 배치를 8로 둡니다. `CUDA out of memory` 가 나면 4로 줄이세요.
EVAL_BATCH = 8

def evaluate_all(m, show=1):
    '''수학추론만 건수를 줄여서 평가한다 (생성이 길어 느리다).

    keep_preds=True — 점수만이 아니라 **모델이 실제로 쓴 답**을 들고 있는다.
    §7-1 에서 맞힌 예와 틀린 예를 직접 보는 데 쓴다.
    '''
    fast = [t for t in EVAL_TASKS if t != "math"]
    res = evaluate_tasks(m, tokenizer, DATA_DIR, tasks=fast, limit=LIMIT,
                         batch_size=EVAL_BATCH, show=show, keep_preds=True) if fast else {}
    if "math" in EVAL_TASKS:
        res.update(evaluate_tasks(m, tokenizer, DATA_DIR, tasks=["math"], limit=LIMIT_MATH,
                                  batch_size=EVAL_BATCH, show=show, keep_preds=True))
    return {t: res[t] for t in EVAL_TASKS if t in res}


def without_preds(res):
    '''파일로 남길 때는 예측 본문을 뺀다 — 점수만 있으면 되고, 파일이 커진다.'''
    return {t: {k: v for k, v in d.items() if k != "preds"} for t, d in res.items()}

t0 = time.time()
before = evaluate_all(model)
print(f"\n학습 전 평가 완료: {(time.time()-t0)/60:.1f}분")
Path(f"{OUT_DIR}/before.json").write_text(json.dumps({"model": MODEL_ID, "results": without_preds(before)}, ensure_ascii=False, indent=2), encoding="utf-8")

> **관찰 포인트 ③** `형식깨짐` 건수를 보세요. 주로 **개체명인식**(JSON 배열), **문장유사도**(소수 하나), **SQL생성**(코드블록 표시를 붙임)에서 나옵니다 —
> Instruct 모델도 "하나만 답하라"는 지시를 종종 어기고 설명을 덧붙입니다. 그리고 점수 자체를 보세요: 문장유사도는 상관이 음수인 경우도 있습니다.
> 파인튜닝이 가장 먼저 고치는 것이 **형식**이고, 그 다음이 태스크 자체입니다.

## 5. LoRA — 무엇을, 왜, 얼마나 학습하나

7.3B 파라미터를 전부 학습하려면 가중치·기울기·옵티마이저 상태까지 파라미터당 16바이트쯤이 필요해 100GB가 넘습니다. **LoRA**는 다른 길을 갑니다.

In [ ]:
import sys
sys.path.insert(0, str(next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                            if (p / "tools" / "diagram.py").exists()) / "tools"))
from diagram import lora_dims, show
show(lora_dims())

- 각 선형층의 가중치 `W`는 그대로 두고, 옆에 **작은 행렬 두 개 `A`(r×k), `B`(d×r)** 를 붙여 그 둘만 학습합니다. `r`(rank)이 8~64면 학습 파라미터가 전체의 1~2%로 줍니다.
- 학습이 끝나면 `B·A`를 `W`에 더해 합칠 수도 있고(merge), 어댑터 파일만 따로 저장해 원본 모델에 갈아 끼울 수도 있습니다.
- 메모리·시간이 크게 줄고, 원본 모델의 능력을 덜 잊습니다(catastrophic forgetting 완화).

`LoraConfig`의 주요 값:

| 인자 | 뜻 | 여기서 |
|---|---|---|
| `r` | 작은 행렬의 rank. 클수록 표현력↑ 파라미터↑ | 16 |
| `lora_alpha` | 스케일. 보통 `r`의 2배 | 32 |
| `lora_dropout` | 어댑터 입력에 드롭아웃 | 0.05 |
| `target_modules` | 어느 층에 붙일지. `"all-linear"`면 모든 선형층(어텐션 q/k/v/o + FFN) | all-linear |
| `task_type` | 모델 종류 | `CAUSAL_LM` |

📚 [PEFT — LoRA 개념](https://huggingface.co/docs/peft/conceptual_guides/lora) · [LoRA 논문](https://arxiv.org/abs/2106.09685)

### 미션 `m-lora` — 어댑터를 어디에, 얼마나 붙일 것인가

위 표의 값을 코드로 옮깁니다. 값 자체는 표에 다 있으니 **어렵지 않습니다.** 채울 것은 **`____` 두 곳**입니다.
중요한 것은 그 다음 셀에서 **무엇이 실제로 학습 대상이 되었는지** 눈으로 확인하는 것입니다.

In [ ]:
quizkit.show("m-lora-1")

In [ ]:
quizkit.show("m-lora-2")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  미션 m-lora-1 · m-lora-2 — 어댑터를 어디에, 얼마나 붙일 것인가
# ══════════════════════════════════════════════════════════════════════
#
# 참고. 같은 설정은 `task5-llm-ft/train.py` 에 있습니다 — 감추지 않고 알려 드립니다.

from peft import LoraConfig, get_peft_model

# 아래 ____ 두 곳만 채우세요. 값은 위 표에 다 있고, 바로 위 셀의 객관식에서 고른 답을 옮기면 됩니다.

def build_lora_config():
    # r=16 이면 A(16×k)·B(d×16) 두 행렬만 학습한다. alpha 는 보통 r 의 2배로 둔다.
    return LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
        task_type=____,                                      # m-lora-1
        target_modules=____,                                 # m-lora-2
    )


def attach_lora(model, config):
    # 원본 가중치는 얼리고(requires_grad=False) LoRA 행렬만 학습 대상으로 남긴다.
    return get_peft_model(model, config)


lora_config = build_lora_config()
model.config.use_cache = False                      # 학습 중에는 KV 캐시를 끈다
peft_model = attach_lora(model, lora_config)        # 원본을 감싸고 LoRA 행렬을 끼운다
peft_model.print_trainable_parameters()

# 어댑터가 실제로 어디에 붙었나 — 이름 끝이 lora_A.default 인 모듈을 센다
lora_names = [n for n, _ in peft_model.named_modules() if n.endswith("lora_A.default")]
print(f"\nLoRA가 붙은 선형층 {len(lora_names)}개. 예:")
for n in lora_names[:7]:
    print("  ", n.replace(".lora_A.default", ""))

#### 확인

> 빈칸을 채우지 않으면 여기서 `NameError: name '____' is not defined` 로 멈춥니다. 정상입니다 —
> 위 셀의 `____` 두 곳을 채우고 그 셀을 다시 실행한 뒤 이 셀을 실행하세요.

In [ ]:
assert lora_config.r == 16, f"r 이 16 이어야 합니다 (지금 {lora_config.r})"
assert lora_config.lora_alpha == 32, f"lora_alpha 가 32 여야 합니다 (지금 {lora_config.lora_alpha})"
assert str(lora_config.task_type) .endswith("CAUSAL_LM"), (
    f'task_type 이 "CAUSAL_LM" 이어야 합니다 (지금 {lora_config.task_type})')
assert len(lora_names) > 0, "LoRA 가 어느 층에도 붙지 않았습니다 — target_modules 를 확인하세요"

_train = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
_total = sum(p.numel() for p in peft_model.parameters())
_pct = _train / _total * 100
assert 0.3 < _pct < 5.0, f"학습 파라미터 비율이 {_pct:.2f}% 입니다 — 1~2% 안팎이어야 정상입니다"

# 원본 가중치는 전부 얼어 있어야 한다
_thawed = [n for n, q in peft_model.named_parameters() if "lora_" not in n and q.requires_grad]
assert not _thawed, f"LoRA 가 아닌 가중치가 학습 대상으로 남아 있습니다: {_thawed[:3]}"

print(f"통과 — 학습 파라미터 {_train/1e6:.1f}M / 전체 {_total/1e9:.2f}B = {_pct:.2f}%")
print(f"        LoRA 가 붙은 선형층 {len(lora_names)}개, 원본 가중치는 전부 얼어 있습니다")

### 퀴즈 `q-lora`

방금 붙인 어댑터가 왜 통하는지 정리해 보는 문제입니다. 보기를 고르면 해설이 열립니다.

In [ ]:
quizkit.show("q-lora")

> **관찰 포인트 ④** `trainable params`가 전체의 몇 %인지 보세요. 1일차 BERT 실습에서는 110M 전부를 학습했습니다. 여기서는 1.2B 모델의 **1~2%** 만 학습합니다. 이 비율이 어댑터 파일 크기(§6)와 GPU 메모리를 결정합니다.

### 5-1. QLoRA — 얼려 둔 가중치를 4비트로

LoRA는 학습하는 파라미터를 줄였지만, **얼려 둔 원본 가중치는 여전히 16비트로 GPU에 올라가 있습니다**.
1.2B 모델이면 약 2.4GB, 7.3B 모델이면 약 15GB입니다. 큰 모델을 작은 GPU에 올릴 때 이것이 벽이 됩니다.

**QLoRA**는 그 얼려 둔 부분만 **4비트로 압축해서** 올립니다([Dettmers et al. 2023](https://arxiv.org/abs/2305.14314)).
학습하는 LoRA 행렬은 그대로 16비트이므로 학습의 정밀도는 유지되고, 메모리만 줍니다. 대신 매 계산마다 4비트를 풀어야 해서 **조금 느려집니다**.

```bash
python task5-llm-ft/train.py --load-4bit --batch-size 2 --grad-accum 8
```

세 가지를 함께 보아야 판단이 됩니다 — **메모리가 얼마나 주는가 / 시간이 얼마나 느는가 / 점수가 떨어지는가**.
아래 셀이 강사 스윕 결과(`results/sweep2/`)에서 같은 모델의 16bit와 4bit를 나란히 놓습니다.

In [ ]:
sw2 = load_sweep(RESULTS, "sweep2")
pair = {"exaone4-1.2b": "16bit LoRA", "exaone4-1.2b-4bit": "4bit QLoRA"}
have = [t for t in pair if len(sw2) and t in set(sw2["tag"])]
if len(have) == 2:
    rows = []
    for tag in pair:
        s = sw2[sw2["tag"] == tag]
        rows.append({"구성": pair[tag],
                     "GPU 최대(GB)": s["peak_gpu_gb"].iloc[0],
                     "학습(분)": s["train_min"].iloc[0],
                     "6태스크 평균 점수": round(s["after"].mean(), 1)})
    t = pd.DataFrame(rows).set_index("구성")
    display(t)
    d_mem = t.loc["4bit QLoRA", "GPU 최대(GB)"] - t.loc["16bit LoRA", "GPU 최대(GB)"]
    d_min = t.loc["4bit QLoRA", "학습(분)"] - t.loc["16bit LoRA", "학습(분)"]
    d_sc = t.loc["4bit QLoRA", "6태스크 평균 점수"] - t.loc["16bit LoRA", "6태스크 평균 점수"]
    print(f"4bit로 바꾸면 메모리 {d_mem:+.2f} GB, 학습 시간 {d_min:+.1f}분, 평균 점수 {d_sc:+.1f}점")
    display(sw2[sw2["tag"].isin(have)].pivot_table(index="task_name", columns="tag", values="after").round(1))
elif len(have) == 1:
    print(f"16bit/4bit 중 하나만 있습니다({have}). 비교는 둘 다 나온 뒤에 가능합니다.")
else:
    print("results/sweep2/ 에 4bit 결과가 아직 없습니다 — 아직 측정 전입니다.")
    print("직접 재 보려면: python task5-llm-ft/train.py --load-4bit --batch-size 2 --grad-accum 8")

## 6. TRL `SFTTrainer`로 학습

`SFTTrainer`는 `messages` 형식 데이터를 받아 **chat template 적용 → 토큰화 → 배치 → 손실 계산**을 대신 해 줍니다.
1일차 실습에서 손으로 짰던 `Dataset`·`collate_fn`·학습 루프가 이 안에 들어 있습니다. 우리가 정하는 것은 `SFTConfig`의 하이퍼파라미터입니다.

| 인자 | 뜻 | 값 | 왜 |
|---|---|---|---|
| `num_train_epochs` | 데이터를 몇 번 훑나 | 1 | 수업 시간 안에 끝나야 함. 1 epoch로도 효과가 뚜렷함 |
| `per_device_train_batch_size` | 한 번에 넣는 예제 수 | 4 | 메모리 |
| `gradient_accumulation_steps` | 몇 번 모아 한 번 갱신하나 | 4 | 유효 배치 = 4×4 = 16 |
| `learning_rate` | 학습률 | 2e-4 | LoRA는 전체 파인튜닝(2e-5)보다 10배쯤 크게 |
| `max_length` | 토큰 길이 상한 | 1024 | 기계독해 지문과 SQL 스키마가 가장 길다 |
| `bf16` | 16비트 연산 | True | 속도·메모리 |
| `gradient_checkpointing` | 중간 활성값을 버리고 다시 계산 | True | 메모리 ↓ (시간 약간 ↑) |
| `packing` | 짧은 예제를 이어붙이기 | False | 태스크 경계를 지킨다 |
| `assistant_only_loss` | **답변 부분에만** 손실 | 템플릿이 지원할 때 | 지시문까지 외우게 하는 건 낭비 (§3-1) |

`assistant_only_loss`는 chat template에 답변 구간 표시(`{% generation %}`)가 있어야 쓸 수 있습니다.
우리가 붙이는 단순 형식에는 마커를 넣어 두었고 Qwen 계열 템플릿에도 있습니다.
EXAONE-4.0·kanana-1.5·Gemma-4의 고유 템플릿에는 없어 **전체 시퀀스**로 학습합니다 — 학습은 되지만 점수가 조금 손해입니다.
기본 모델 A.X-4.0-Light 는 `<|im_end|>` 를 쓰는 계열이라 **답변 부분에만** 손실을 겁니다.
이런 모델에는 `train.py --plain-template`로 우리 단순 형식을 강제해 답변 손실을 켤 수 있습니다(§11 더 해보기).
그 차이를 실제로 재 본 대조 실험이 `results/ablation/`에 있습니다(이전 구성 기준, `bert_vs_llm.ipynb` §2-1).

📚 [TRL — SFTTrainer](https://huggingface.co/docs/trl/sft_trainer) · [SFTConfig 전체 인자](https://huggingface.co/docs/trl/sft_trainer#trl.SFTConfig)

In [ ]:
from train import load_dataset, supports_assistant_only     # task5-llm-ft/train.py 의 함수를 그대로 쓴다

train_ds = load_dataset(TRAIN_FILE)
print(f"학습 데이터 {len(train_ds)}건 — {pd.Series(train_ds['task']).value_counts().reindex(MAIN_TASKS).to_dict()}")

assistant_only = supports_assistant_only(tokenizer)
print("손실 계산 방식:", "답변 부분에만 손실" if assistant_only else "전체 시퀀스 (이 모델의 템플릿은 답변 구간 표시가 없음)")

In [ ]:
from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    output_dir=OUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    max_length=1024,
    logging_steps=10,
    save_strategy="no",                 # 끝에 한 번만 저장 (중간 체크포인트 불필요)
    bf16=True,
    gradient_checkpointing=True,
    packing=False,
    report_to=[],
    seed=42,
    assistant_only_loss=assistant_only,
)
from common import MidTrainEvalCallback, attach_rich_progress   # noqa: E402

mid_eval = MidTrainEvalCallback(tokenizer)      # 10%마다 태스크당 20건을 실제로 생성·채점한다 (경향용)
trainer = SFTTrainer(
    model=peft_model,                   # LoRA가 이미 붙은 모델을 넘긴다 (peft_config는 넘기지 않는다)
    args=sft_config,
    train_dataset=train_ds.remove_columns("task"),
    processing_class=tokenizer,
    callbacks=[mid_eval],
)
# 진행률 표를 넓힌다 — Step·Training Loss 뿐이던 것을 진행률·토큰 정확도·학습률·GPU 메모리·경과/남은 시간,
# 그리고 중간 점검이 있던 스텝엔 태스크별 검증 점수까지 한 표에.
attach_rich_progress(trainer, mid_eval)
eff_batch = sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps
steps = len(trainer.get_train_dataloader()) // sft_config.gradient_accumulation_steps
print(f"학습 스텝 수: 약 {steps} ({len(train_ds)}건 ÷ 유효배치 {eff_batch})")

학습을 시작합니다. 걸리는 시간은 모델과 GPU에 따라 다릅니다 — §0의 표에 강사가 잰 값이 있습니다(몇 분~10여 분).
기다리는 동안 아래 영상을 보세요 — **"How Large Language Models Work"** (3Blue1Brown). 오늘 배운
프롬프트 → 토큰 → 다음 토큰 예측을 애니메이션으로 보여 줍니다. 어제 강의자료(1일차 덱) §29에도 있던 그 영상입니다.

In [ ]:
from IPython.display import YouTubeVideo, HTML, display
display(YouTubeVideo("HnvitMTkXro", width=640, height=360))
_qr_b64 = "iVBORw0KGgoAAAANSUhEUgAAASwAAAEsAQMAAABDsxw2AAAABlBMVEX///8AAABVwtN+AAAACXBIWXMAAA7EAAAOxAGVKw4bAAAB1klEQVRoge2avY3DMAyFaaRI6RE8ikezR/MoGSGliyA6iY8UgzsESEPHBzw2UcTPbh74I8oin9lSzPTffL/U5T6UUhcyue9GLBm7qb9uyrWUbdTtJxSadvwSy8eqF9LsEMu8uqiqlQexIzHTqIYMsf+CYVFDjNiBmOe35p3v4tZDTIjlY73WN2m2sUlzfcrytiUgloCFIVI6Jm+M2Pcw5Ldi7TGs7qyVN00fxNIxXW8i0GhWhxaXgsha1UUsG2ulBGeTokXfFirRoJiKRewU2KCbHesPaojpDrFkDI1ZE+smztshBeXG0yCxTAwiPVwsQcZbPYhayET/RiwHw1CleK0fUdnRcU32BkQWsa9j2pht8A3RG5jUTVPLb8QSMXh1lmKHlGiPJfIbsVRMq8zsj8wq0GsPto4x4yKWilmtf/Fq0W+xY6mM2Dmwq02DRROdtccLGubm8jRILBWDNzRq5/pozKAjsVQszMfCaLRwQsSYhdhJsMW0s88hUJUG9GN+nyVCLBmDXpdozKy4lBI8sXTMB1l6ZtQBo91nqfXGjNgxmN3txnAeIfVyk0LsNJhfAUPKEkYsH/P8JjZmiW8a/wzniWVhXZH+IRCG82bL75aA2Pewz+wHlQ3Ex0XC2IkAAAAASUVORK5CYII="
display(HTML('<div style="margin-top:8px;display:flex;align-items:center;gap:14px">'
             f'<img src="data:image/png;base64,{_qr_b64}" width="130" height="130" '
             'style="border:1px solid #ddd;border-radius:6px">'
             '<div>휴대폰 카메라로 이 QR을 비추면 같은 영상이 열립니다.<br>'
             '<code>https://youtu.be/HnvitMTkXro</code></div></div>'))

진행 막대의 `loss`가 내려가는지도 함께 지켜보세요. 아래 셀을 실행하면 학습이 시작됩니다.

In [ ]:
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
train_out = trainer.train()
train_sec = time.time() - t0
peak_gb = torch.cuda.max_memory_allocated() / 1024**3
print(f"\n학습 완료: {train_sec/60:.1f}분 | GPU 최대 {peak_gb:.1f} GB | 최종 손실 {train_out.training_loss:.4f}")

In [ ]:
# 어댑터 저장 — 원본 모델은 저장하지 않는다. 어댑터 + 토크나이저(=템플릿)만 있으면 재현할 수 있다.
trainer.save_model(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
!du -sh {OUT_DIR}/adapter_model.safetensors
print("저장 위치:", OUT_DIR)

In [ ]:
import matplotlib.pyplot as plt
from compare import setup_korean_font
setup_korean_font()

log = [(x["step"], x["loss"]) for x in trainer.state.log_history if "loss" in x]
xs, ys = zip(*log)
plt.figure(figsize=(7, 3.2))
plt.plot(xs, ys, marker="o", ms=3)
plt.xlabel("step"); plt.ylabel("train loss"); plt.title(f"{MODEL_ID.split('/')[-1]} — 학습 손실 (1 epoch)")
plt.grid(alpha=.3); plt.tight_layout(); plt.show()
print(f"첫 손실 {ys[0]:.3f} → 마지막 손실 {ys[-1]:.3f}")

> **관찰 포인트 ⑤** 손실은 처음 수십 스텝에서 급히 떨어지고 그 뒤 완만해집니다. 초반 급락은 **형식**(라벨 집합, JSON, `#### 답` 줄, 종료 토큰)을 배우는 구간, 완만한 구간이 **내용**을 배우는 구간이라고 볼 수 있습니다.
> 어댑터 파일이 몇 MB인지도 확인하세요 — 원본 모델의 1~2%입니다.

## 7. 학습 후 평가 — 전후 비교

같은 모델 객체에 어댑터가 붙어 있으므로, **어댑터를 잠깐 끄면 학습 전 모델**이 됩니다(`disable_adapter`).
메모리를 두 배로 쓰지 않고 전후를 나란히 볼 수 있습니다 — 데모 페이지(§9)도 같은 방법을 씁니다.

In [ ]:
# 추론 모드로 전환
peft_model.eval()
peft_model.config.use_cache = True
peft_model.gradient_checkpointing_disable()

for task, payload in EXAMPLES:
    with peft_model.disable_adapter():
        b = ask(task, peft_model, **payload)
    a = ask(task, peft_model, **payload)
    print(f"[{task}] {TASKS[task]['name']}")
    print(f"   학습 전: {b[:120]!r}")
    print(f"   학습 후: {a[:120]!r}\n")

In [ ]:
t0 = time.time()
after = evaluate_all(peft_model)
print(f"\n학습 후 평가 완료: {(time.time()-t0)/60:.1f}분")
Path(f"{OUT_DIR}/after.json").write_text(json.dumps({"model": MODEL_ID, "adapter": OUT_DIR, "results": without_preds(after)}, ensure_ascii=False, indent=2), encoding="utf-8")

In [ ]:
from IPython.display import Markdown
from common import main_metric
import numpy as np

display(Markdown(results_table({"학습 전": before, "학습 후 (LoRA 1 epoch)": after}, tasks=EVAL_TASKS)))

tasks = [t for t in EVAL_TASKS if t in before and t in after]
assert tasks, "학습 전/후 결과가 모두 있는 태스크가 없습니다 — §4·§7 평가 셀을 먼저 실행하세요"
names = [f"{TASKS[t]['name']}\n({TASKS[t]['metric']})" for t in tasks]
b = [main_metric(t, before[t]) for t in tasks]
a = [main_metric(t, after[t]) for t in tasks]
x = np.arange(len(tasks)); w = 0.38
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.bar(x - w/2, b, w, label="학습 전", color="#9e9e9e")
ax.bar(x + w/2, a, w, label="학습 후", color="#e07b39")
for i in range(len(tasks)):
    ax.text(x[i]-w/2, max(b[i], 0)+1, f"{b[i]:.0f}", ha="center", fontsize=8)
    ax.text(x[i]+w/2, max(a[i], 0)+1, f"{a[i]:.0f}", ha="center", fontsize=8, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(names, fontsize=9); ax.set_ylim(min(0, min(b)-5), 105); ax.set_ylabel("점수")
ax.set_title(f"{MODEL_ID.split('/')[-1]} — 학습 전/후"); ax.legend(loc="upper left"); ax.grid(axis="y", alpha=.3)
plt.tight_layout(); plt.show()
print(f"평균: 학습 전 {np.mean(b):.1f} → 학습 후 {np.mean(a):.1f}  (+{np.mean(a)-np.mean(b):.1f})")
print("형식 깨짐 합계: 학습 전", sum(before[t].get("broken_format", 0) for t in tasks),
      "→ 학습 후", sum(after[t].get("broken_format", 0) for t in tasks))

> **관찰 포인트 ⑥** 어느 태스크가 가장 많이 올랐고, 어느 태스크가 여전히 낮은가요?
> 보통 **주제분류·문장유사도**는 크게 오르고, **개체명인식**은 JSON 형식과 경계를 함께 맞춰야 해서 어렵습니다.
> **수학추론·SQL생성**은 형식은 금방 배우지만 정답률은 모델 크기의 영향을 크게 받습니다 — 1~2B 모델이 여러 단계 계산을 끝까지 맞히기는 쉽지 않습니다.
> **형식 깨짐**이 거의 0으로 줄었는지도 보세요 — 그것이 1 epoch SFT가 가장 확실히 해내는 일입니다.

### 7-1. 맞힌 예와 틀린 예 — 점수 뒤에 있는 실제 답

막대그래프는 평균입니다. **모델이 실제로 뭐라고 썼는지**는 직접 봐야 압니다.
태스크마다 **잘 맞힌 두 건을 먼저, 크게 틀린 두 건을 그다음에** 놓았습니다.

틀린 것만 보면 "이 모델은 못 쓰겠다"로 읽히기 쉽지만, 주제분류 85점이란 스무 개 중 열일곱을
맞혔다는 뜻입니다. 맞힌 쪽을 먼저 보셔야 그 감이 잡히고, 그다음에 **어디서 무너지는지**가 보입니다.

`채점` 칸은 예제 하나의 점수입니다. 주제분류·수학추론은 맞다/틀리다뿐이고,
개체명·기계독해·SQL은 겹치는 정도라서 부분 점수가 나옵니다.

In [ ]:
from common import examples_table
from IPython.display import display, Markdown

# 어느 태스크를 볼지 — 바꿔 가며 실행해 보세요
SHOW_TASK = "tc"        # tc · ner · mrc · sts · sql · math

rows = read_jsonl(f"{DATA_DIR}/eval_{SHOW_TASK}.jsonl")[:LIMIT_MATH if SHOW_TASK == "math" else LIMIT]
display(Markdown(f"#### {TASKS[SHOW_TASK]['name']} — 학습 **후** 모델"))
display(examples_table(SHOW_TASK, after[SHOW_TASK]["preds"], rows, 2))
display(Markdown(f"#### {TASKS[SHOW_TASK]['name']} — 학습 **전** 모델 (같은 문제)"))
display(examples_table(SHOW_TASK, before[SHOW_TASK]["preds"], rows, 2))

> **관찰 포인트 ⑥-1** 학습 전 표의 `모델 예측` 칸을 보세요. 답을 몰라서 틀린 것이 아니라
> **형식을 안 지켜서** 틀린 것이 많습니다 — 라벨 하나만 쓰라는데 설명을 붙이고, 숫자 하나만
> 쓰라는데 문장을 씁니다. 학습 후 표에서 그 군더더기가 사라진 것이 보이면,
> 1 epoch SFT 가 가장 먼저 해내는 일이 무엇인지 눈으로 확인한 것입니다.

### 퀴즈 `q-sql`

두 지표가 왜 이렇게 갈리는지 생각해 보는 문제입니다. 보기를 고르면 해설이 열립니다.

In [ ]:
quizkit.show("q-sql")

## 8. BERT·T5와 비교 — 같은 데이터, 같은 평가셋

1일차 방식으로 **똑같은 학습 예제·똑같은 평가셋·똑같은 채점 함수**를 쓰면 어떻게 될까요?
`bert_baseline.py`(인코더)와 `t5_baseline.py`(인코더-디코더)가 그 일을 합니다.

- `--mode budget` — LLM이 쓴 것과 **같은 소규모 학습셋**으로 3 epoch. *"같은 데이터를 주면 누가 더 잘 배우나"*
- `--mode full` — 공식 train **전체**로 1 epoch. *"BERT가 제 실력을 다 내면 얼마나 차이 나나"*

수업에서는 빠른 두 태스크(주제분류·문장유사도)만 budget 모드로 돌려 봅니다(각 30초~1분). 나머지는 강사 스윕 결과를 봅니다.

In [ ]:
for task in ["tc", "sts"]:
    !{PY} task5-llm-ft/bert_baseline.py --task {task} --mode budget --model klue/roberta-base --limit {LIMIT} --save {OUT_DIR}/bert-{task}-budget.json --out output/nb/bert-{task}

In [ ]:
rows = []
for task in ["tc", "sts"]:
    p = Path(f"{OUT_DIR}/bert-{task}-budget.json")
    if not p.exists() or task not in after:
        continue
    r = json.loads(p.read_text(encoding="utf-8"))
    s = r["results"][task]; tr = r["train"]
    rows.append({"태스크": TASKS[task]["name"], "지표": TASKS[task]["metric"],
                 "BERT (klue/roberta-base, 같은 학습셋 3 epoch)": round(main_metric(task, s), 1),
                 f"LLM ({MODEL_ID.split('/')[-1]} LoRA 1 epoch)": round(main_metric(task, after[task]), 1),
                 "BERT 학습(분)": round(tr["train_seconds"]/60, 1), "BERT 추론(초)": s["sec"], "LLM 추론(초)": after[task]["sec"]})
if rows:
    display(pd.DataFrame(rows).set_index("태스크"))
else:
    print("BERT 기준선 결과가 없습니다 — 위 셀이 정상적으로 끝났는지 확인하세요.")

아래는 강사가 미리 돌린 **세 방식 비교**입니다(평가 각 300건). 결과 파일이 저장소에 함께 들어 있어 GPU 없이도 그려집니다.
BERT 칸의 **수학추론·SQL생성이 비어 있는 것은 결과가 없어서가 아니라, 그 구조로는 만들 수 없기 때문**입니다.

In [ ]:
from compare import load_bert, load_t5, TASK_ORDER

try:
    show = pd.DataFrame(index=[t for t in TASK_ORDER])
    bert = load_bert(RESULTS)
    if len(bert):
        ref = bert[bert["short"] == "roberta-base"]
        for mode, label in [("budget", "BERT budget (같은 학습셋)"), ("full", "BERT full (공식 train 전체)")]:
            show[label] = ref[ref["mode"] == mode].set_index("task")["score"].reindex(show.index)
    t5 = load_t5(RESULTS)
    if len(t5):
        ref5 = t5[t5["short"] == "pko-t5-base"]
        for mode, label in [("budget", "T5 budget (같은 학습셋)"), ("full", "T5 full (공식 train 전체)")]:
            sub = ref5[ref5["mode"] == mode]
            if len(sub):
                show[label] = sub.set_index("task")["score"].reindex(show.index)
    sw2 = load_sweep(RESULTS, "sweep2")
    for tag in ["ax4-light-7b", "exaone4-1.2b", "kanana15-2.1b-instruct", "qwen35-2b"]:
        sub = sw2[sw2["tag"] == tag] if len(sw2) else sw2
        if len(sub):
            show[f"LLM {sub['model'].iloc[0]}"] = sub.set_index("task")["after"].reindex(show.index)

    show = show.dropna(axis=1, how="all")
    if show.empty or not len(show.columns):
        print("비교할 결과 파일이 아직 없습니다 (results/bert, results/t5, results/sweep2).")
    else:
        gen_only_names = [TASK_NAME[t] for t in GEN_ONLY_TASKS]
        show.index = [TASK_NAME[t] for t in show.index]
        numeric = show.astype(float)
        pretty = numeric.round(1).astype(object)          # 숫자 칸과 '구조상 불가'를 한 표에 섞기 위해
        for col in pretty.columns:
            if col.startswith("BERT"):
                for name in gen_only_names:
                    if name in pretty.index:
                        pretty.loc[name, col] = "구조상 불가"
        display(pretty.where(pretty.notna(), "—"))

        ax = numeric.plot.bar(figsize=(11, 3.8), width=0.82, rot=0)
        ax.set_ylim(0, 100); ax.set_ylabel("점수")
        ax.set_title("BERT(인코더) · pko-T5(인코더-디코더) · GPT 계열(디코더) — 같은 평가셋")
        ax.grid(axis="y", alpha=.3); ax.legend(fontsize=8, ncol=2)
        plt.tight_layout(); plt.show()
except Exception as e:
    print("비교표를 그릴 수 없습니다:", type(e).__name__, e)

> **관찰 포인트 ⑦ — 세 가지 결론.**
> 1. **같은 (작은) 데이터**를 주면 대체로 LLM이 낫습니다 — 특히 문장유사도처럼 "이해"가 필요한 태스크에서. 사전학습에서 이미 많은 것을 알고 있기 때문입니다.
> 2. **BERT가 전체 데이터를 다 쓰면** 소형 LLM과 비슷하거나 앞섭니다 — 특히 개체명인식처럼 데이터가 풍부하고 라벨이 정해진 태스크에서. 그리고 **추론 속도는 BERT가 압도적**입니다(한 번의 forward vs 토큰 수만큼 반복 생성).
> 3. **오른쪽 두 칸(수학추론·SQL생성)이 이번 개편의 요지입니다.** BERT는 점수가 낮은 것이 아니라 **출전 자체가 불가능**합니다.
>    정해진 라벨을 대량으로 붙이는 일이라면 BERT가 여전히 실용적인 선택이고, 답을 *만들어야* 하는 일이라면 생성 모델밖에 답이 없습니다.
>
> 구현 관점의 차이(코드 양, 새 태스크 추가 비용, 설명 가능성, 배포)는 `bert_vs_llm.ipynb`에 표로 정리해 두었습니다.

## 9. 내 모델 데모 페이지

`serve.py`는 Flask로 작은 웹 페이지를 띄웁니다. 태스크를 고르고 문장을 넣으면 **학습 전 답과 학습 후 답을 나란히** 보여 줍니다(어댑터를 켜고 끄는 방식, §7과 같음).

1. 먼저 노트북이 잡고 있는 GPU 메모리를 풀어 줍니다(아래 셀).
2. **켜기 셀**(`demo_start(...)`)을 실행합니다 — 배경에서 서버를 띄우므로 셀이 곧 끝나고 노트북을 계속 쓸 수 있습니다. 7B 모델을 읽는 데 1~2분 걸립니다. 다 보고 나면 **끄기 셀**(`demo_stop(9005)`)로 내립니다.
3. 브라우저에서 <http://localhost:9005> 를 엽니다. 다른 자리에서 접속하려면 `localhost` 대신 그 PC의 IP를 씁니다.

터미널에서 띄워도 됩니다:

```bash
python task5-llm-ft/serve.py --model skt/A.X-4.0-Light --adapter output/nb/A.X-4.0-Light --port 9005 --tasks tc,ner,mrc,sts,math,sql
```

In [ ]:
# GPU 메모리 해제 (데모 서버를 같은 GPU에서 띄우기 전에)
# 나중에 다시 돌아와 이 셀만 실행해도 되게 — 변수가 없으면 조용히 넘어간다
for _n in ("trainer", "peft_model", "model"):
    globals().pop(_n, None)
gc.collect(); torch.cuda.empty_cache()
print(f"GPU 사용 {torch.cuda.memory_allocated()/1024**3:.1f} GB")
print(f"\n터미널에서:\n  python task5-llm-ft/serve.py --model {MODEL_ID} --adapter {OUT_DIR} --port 9005 --tasks {','.join(MAIN_TASKS)}")

In [ ]:
# ▶ 켜기 — 배경에서 서버를 띄웁니다. 7B 모델을 읽는 데 1~2분 걸리고, 그동안 점이 찍힙니다
from common import demo_start
demo_start([PY, "task5-llm-ft/serve.py", "--model", MODEL_ID, "--adapter", OUT_DIR,
            "--port", "9005", "--tasks", ",".join(MAIN_TASKS)], port=9005, name="LLM 데모", wait=240)

In [ ]:
# ■ 끄기 — 다 보고 나면 실행하세요 (GPU 를 돌려줍니다). 다시 보려면 켜기 셀을 다시 실행하면 됩니다
from common import demo_stop
demo_stop(9005, name="LLM 데모")

**넣어 볼 것들** — 학습 데이터에 없는 종류를 골라 보세요.

- 주제분류: 뉴스 제목이 아닌 문장 — *"오늘 점심에 김치찌개를 먹었다"*. 두 주제에 걸친 제목 — *"삼성전자, 반도체 공장에 30조 투자"* (경제? IT과학?).
- 개체명인식: 최근 인물·기관 — *"이재명 대통령은 12월 3일 대전 ETRI를 방문했다."* 학습 데이터(2016~2020 뉴스)에 없는 이름도 잡는지.
- 기계독해: 답이 지문에 **없는** 질문을 해 보세요. 모델이 지어내는지(hallucination), 학습 후엔 어떻게 달라지는지. **오늘 오전 실습4A의 BERT는 이 질문에 지문 밖 답을 낼 수 없었습니다** — 그 차이가 여기서 드러납니다.
- 문장유사도: 같은 뜻 다른 표현 vs 단어는 겹치는데 뜻이 다른 쌍 — *"고양이가 개를 쫓는다" / "개가 고양이를 쫓는다"*.
- 수학추론: 두세 단계 계산이 필요한 문제. 풀이는 그럴듯한데 마지막 답만 틀리는 경우를 찾아 보세요.
- SQL생성: 데모의 스키마에 **없는** 테이블을 물어보면 어떻게 되나요?

> **관찰 포인트 ⑧** 학습 후 모델이 틀리는 예를 찾아 보세요. 그리고 학습 **전** 모델은 그 예에서 어땠나요?
> 파인튜닝이 "형식"을 고친 것인지 "능력"을 더한 것인지 구분해 보세요.

## 10. 사례 연구 — "멈추지 못한 모델"

강사가 여러 모델을 같은 코드로 돌리다가 두 번 겪은 일입니다. 점수표만 보면 "이 모델은 못 배운다"로 끝났을 문제였고, **출력을 직접 읽어서** 원인을 찾았습니다.
(둘 다 이전 구성의 스윕에서 겪은 일이지만, 원인과 조치는 지금 코드에 그대로 들어 있습니다.)

### 사례 A. 학습된 적 없는 종료 토큰

기계독해 EM이 학습 후에도 바닥에 머물렀습니다. 출력을 보니 답은 맞게 쓰고 나서 **멈추지 않고** 쓰레기 토큰(`\x00` 바이트 등)을 `max_new_tokens`까지 뱉었습니다.
원인: chat template은 특정 토큰으로 턴을 끝내라고 하는데, 그 모델의 출력층에서 **그 토큰의 행이 다른 특수 토큰들과 완전히 같은 값**(사전학습에서 한 번도 안 쓰인 초기값)이었습니다.
softmax에서 똑같은 행들은 항상 같은 확률을 받으므로 **그 토큰만 골라 내는 법을 파인튜닝으로도 배울 수 없습니다**.
조치: `common.check_end_token`이 모델을 올릴 때 이것을 검사해, Instruct 모델이면 템플릿의 종료 토큰을 실제로 학습된 `eos_token`으로 바꾸고, Base 모델이면 우리 단순 형식으로 바꿉니다.
고치기 전에는 "못 배우는 모델"로 보이던 둘이, 고친 뒤에는 같은 크기 모델 중 상위권이 되었습니다. 상세는 `bert_vs_llm.ipynb`.

### 사례 B. 모델마다 다른 종료 토큰 이름

개체명인식 F1이 학습 후에 오히려 **떨어졌습니다**(재현율이 크게 낮아짐). 학습은 정상이었고 모델은 자기 종료 토큰으로 턴을 끝내는 법을 잘 배웠습니다.
문제는 **평가 코드**였습니다 — 생성을 멈출 토큰 목록이 `<|im_end|>`, `<|eot_id|>` 같은 **알려진 이름만** 담고 있어 그 모델의 종료 토큰이 빠졌고,
모델은 답을 마친 뒤 다음 턴을 계속 지어냈습니다. 그 뒤에 붙은 가짜 개체명들이 정밀도를 깎았습니다.
조치: `common.stop_token_ids`가 종료 토큰을 이름으로 고르지 않고 (1) `eos_token`, (2) **chat template이 실제로 답변 뒤에 붙이는 토큰**, (3) 모델의 `generation_config.eos_token_id` 세 곳에서 모아 씁니다.

두 사례의 교훈:
- **점수표 대신 출력을 읽어라.** `evaluate.py --show 3`은 그래서 있습니다.
- **"형식을 배웠다" ≠ "멈추는 것을 배웠다".** 답변 종료 토큰은 학습 데이터에도, 생성 정지 조건에도 있어야 합니다.
- 새 모델을 붙일 때는 §3의 "마지막 토큰 3개"와 아래 정지 토큰 목록을 먼저 확인하세요. §3-2의 세 가지 점검도 같은 종류의 방어입니다.

In [ ]:
from common import stop_token_ids
eid = end_of_turn_token_id(tokenizer)
print("템플릿이 답변 뒤에 붙이는 토큰 :", repr(tokenizer.convert_ids_to_tokens(eid)) if eid is not None else None)
ids = stop_token_ids(tokenizer)                      # 모델 없이 (1)(2)만; 모델을 넘기면 generation_config도 본다
print("생성을 멈출 토큰들            :", [(i, tokenizer.convert_ids_to_tokens(i)) for i in ids])
print("종료 토큰 점검 결과(§4)       :", end_info)

## 11. 더 해보기 · 참고 자료

**더 해보기** (시간이 남거나 과제로)

1. **평가를 전체 건수로 다시 해 본다.** 노트북은 태스크당 100건만 봤습니다. 수업 시간 안에
   끝나야 하기 때문입니다. 태스크당 300건으로 다시 재면 숫자가 얼마나 달라지나요?
   특히 **수학추론**을 보세요 — 건수를 바꿨을 때 가장 크게 움직이는 태스크입니다.

   ```bash
   # 학습 전 (어댑터 없이) — 몇 분 걸립니다
   python task5-llm-ft/evaluate.py --limit 300 --tasks tc,ner,mrc,sts,sql,math --save output/before300.json
   # 학습 후 (내가 만든 어댑터로)
   python task5-llm-ft/evaluate.py --limit 300 --tasks tc,ner,mrc,sts,sql,math \
       --adapter output/nb/A.X-4.0-Light --save output/after300.json
   ```

   강사가 같은 것을 미리 재 둔 값이 `docs/LECTURE-FACTS.md` 에 있습니다. **내 값과 맞는지** 대조해 보세요.

2. **에폭을 늘려 본다.** `SFTConfig(num_train_epochs=2)` 또는 터미널에서 `python task5-llm-ft/train.py --epochs 3 --out output/exaone-3ep`.
   1 epoch보다 얼마나 더 오르는지, 시간은 몇 배가 드는지 비교하세요. 강사가 같은 모델을 1/2/3 epoch로 돌린 결과가 아래 셀에 나옵니다.
3. **`--plain-template`** — 모델의 고유 대화 형식 대신 단순 형식을 강제합니다(§3-1, §6).
   기본 모델(A.X-4.0-Light)은 답변 부분에만 손실을 걸 수 있지만, EXAONE-4.0처럼 그러지 못하는 모델은 전체 시퀀스로 학습합니다.
   `python task5-llm-ft/train.py --model LGAI-EXAONE/EXAONE-4.0-1.2B --plain-template --out output/exaone-plain` 으로 학습하고
   `evaluate.py --model LGAI-EXAONE/EXAONE-4.0-1.2B --plain-template --adapter output/exaone-plain` 으로 재 보세요.
   어느 태스크가 가장 많이 달라지나요?
4. **`--load-4bit` (QLoRA)** — §5-1. `python task5-llm-ft/train.py --load-4bit --batch-size 2 --grad-accum 8`.
   GPU 최대 사용량·학습 시간·점수 셋을 모두 적어 두고 16bit와 비교하세요.
5. **태스크를 빼고 학습해 본다.** `load_dataset(TRAIN_FILE, tasks=["tc","ner","mrc","sts"])` 로 네 태스크만 학습한 뒤,
   **학습하지 않은** 수학추론·SQL생성 점수가 어떻게 되는지 보세요. 반대로 수학추론만 빼고 학습하면 다른 태스크가 좋아지나요(전이/간섭)?
5. **T5와 같은 태스크로 붙여 본다.** `python task5-llm-ft/t5_baseline.py --task math --mode budget` 처럼 pko-T5로 같은 학습셋·같은 평가셋을 돌려
   §8 표에 한 줄 더합니다. 인코더-디코더가 디코더 전용보다 유리한 태스크가 있나요?
6. **`MODEL_ID`를 바꿔 본다.** Base 모델(`Qwen/Qwen3.5-0.8B-Base`)로 바꾸면 §3에서 단순 형식이 붙고, §4에서 점수가 거의 바닥이며, §7에서 Instruct에 근접합니다.
7. **자기 도메인 데이터를 섞는다.** 문장 50개를 `messages` 형식으로 만들어 학습셋에 섞어 보세요 — `build_dataset.py`의 `pack()` 형식을 따르면 됩니다.
8. **터미널로 전 과정을 재현한다.** `train.py --inspect` → `train.py` → `evaluate.py --show 3` → `serve.py`. 여러 모델을 자동으로 돌리는 `sweep.sh`도 있습니다.

In [ ]:
# 더 해보기 1 — 에폭 실험 결과 (강사 스윕, results/sweep2/)
ep_tags = {"exaone4-1.2b": "1 epoch", "exaone4-1.2b-2ep": "2 epoch", "exaone4-1.2b-3ep": "3 epoch"}
sw2 = load_sweep(RESULTS, "sweep2")
have = [t for t in ep_tags if len(sw2) and t in set(sw2["tag"])]
if len(have) >= 2:
    rows = []
    for tag in ep_tags:
        s = sw2[sw2["tag"] == tag] if len(sw2) else sw2
        if not len(s):
            continue
        row = {"설정": ep_tags[tag], "학습(분)": s["train_min"].iloc[0]}
        row.update({TASK_NAME[t]: v for t, v in zip(s["task"], s["after"])})
        row["평균"] = round(s["after"].mean(), 1)
        rows.append(row)
    display(pd.DataFrame(rows).set_index("설정").round(1))
else:
    print("에폭 실험 결과(results/sweep2/ exaone4-1.2b-2ep, -3ep)가 아직 없습니다.")
    print("직접 해 보려면: python task5-llm-ft/train.py --epochs 3 --out output/exaone-3ep")

**참고 자료**

- 이 저장소: `task5-llm-ft/README.md`(명령 모음), `bert_vs_llm.ipynb`(모델별·구조별 비교), `DEMO-SCRIPT.md`(데모 진행 대본)
- ratsgo, *Do it! BERT와 GPT로 배우는 자연어처리* — [nlpbook](https://ratsgo.github.io/nlpbook/) (실습 구조의 원형)
- Hugging Face LLM Course — [Ch.11 Supervised Fine-Tuning](https://huggingface.co/learn/llm-course/chapter11/1) · [Chat Templates](https://huggingface.co/docs/transformers/chat_templating)
- TRL — [SFTTrainer](https://huggingface.co/docs/trl/sft_trainer) · PEFT — [LoRA](https://huggingface.co/docs/peft/conceptual_guides/lora)
- 논문: [LoRA](https://arxiv.org/abs/2106.09685) (Hu et al., 2021) · [QLoRA](https://arxiv.org/abs/2305.14314) (Dettmers et al., 2023) · [FLAN](https://arxiv.org/abs/2109.01652) (Wei et al., 2022) · [InstructGPT](https://arxiv.org/abs/2203.02155) (Ouyang et al., 2022) · [KLUE](https://arxiv.org/abs/2105.09680) (Park et al., 2021) · [GSM8K](https://arxiv.org/abs/2110.14168) (Cobbe et al., 2021) · [Spider](https://arxiv.org/abs/1809.08887) (Yu et al., 2018)
- 모델 카드: [A.X-4.0-Light](https://huggingface.co/skt/A.X-4.0-Light) · [EXAONE-4.0-1.2B](https://huggingface.co/LGAI-EXAONE/EXAONE-4.0-1.2B) · [kanana-1.5-2.1b-instruct](https://huggingface.co/kakaocorp/kanana-1.5-2.1b-instruct-2505) · [Qwen3.5-2B](https://huggingface.co/Qwen/Qwen3.5-2B) · [Llama-3.2-3B-Instruct](https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct) · [Gemma-4-E2B-it](https://huggingface.co/google/gemma-4-E2B-it) · [pko-t5-base](https://huggingface.co/paust/pko-t5-base)
- 데이터: [KLUE](https://klue-benchmark.com) · [KorQuAD 1.0](https://korquad.github.io/KorQuad%201.0/) · [GSM8K-ko](https://huggingface.co/datasets/kuotient/gsm8k-ko) · [Spider-ko](https://huggingface.co/datasets/huggingface-KREW/spider-ko)

---
*ETRI 언어지능연구실 · AI아카데미 A4021 (2026-09-03/04). 출처 표시: 위 데이터셋·모델·문서는 각 라이선스에 따라 교육 목적으로 인용합니다.*